# 07 · Q2 : l'effet des promotions

**Projet 4 : Analyse des données de vente et retours** · Sprint 4 · US18 et US36

**Question de recherche Q2** : *les promotions sont-elles efficaces ? Augmentent-elles les ventes, et à quel coût ?*

**Objectif de l'énoncé servi par ce notebook** : évaluer l'efficacité des promotions et ce qu'elles coûtent.

**Le cadre d'analyse** : les chiffres de **situation** (coût des remises, seuils, leviers) portent sur le dernier exercice, **2021**, et sont suivis sur les **trois exercices** : **2019**, l'année de référence où tous les magasins gagnaient de l'argent, **2020** et **2021**. Les **mécanismes** (effet des promotions sur les ventes, tests) sont mesurés sur les trois années réunies, puis vérifiés par exercice. Les trois canaux sont suivis, avec l'accent sur les magasins.

**Ce que ce notebook reprend de `06_q1_facteurs_rentabilite`** : au niveau magasin et mois, la part des lignes en promotion va avec un résultat d'exploitation plus faible (corrélation de Spearman −0,44) et un chiffre d'affaires plus faible (−0,21). `06` laissait ouverte la question du sens de la cause : on la traite ici (section 9).

| Étape | Ce qu'on analyse | Question posée |
|---|---|---|
| 1. Chargement | Ventes avec variables, table commune ventes et charges, charges, catalogue | |
| 2. Portrait | Les promotions, leur calendrier, leur étendue, le taux de remise réellement appliqué, le magasin de Cambridge | Comment les promotions sont-elles distribuées ? |
| 3. Évolution 2019 → 2021 | Nombre, taux annoncé, taux effectif, part des lignes ; âge et gamme des produits promus | Les promotions ont-elles suivi la descente en gamme et le vieillissement de l'assortiment ? |
| 4. Le piège | Promotion et saison sont confondues | Pourquoi une simple comparaison ne suffit pas ? |
| 5. Comparaison naïve | Avec et sans promotion, tous mois confondus, puis par exercice | Que dit le calcul direct, et pourquoi il est biaisé ? |
| 6. Angle A | Les 14 jours avant et après chaque changement de promotion | Les ventes bougent-elles quand la promotion change ? |
| 7. Angle B | Le magasin de Cambridge, seul à suivre un autre calendrier | Un magasin plus promu que les autres le même mois vend-il plus ? |
| 8. Angle C | Lignes remisées et non remisées du même produit, le même mois | Une ligne remisée est-elle plus grosse ? |
| 9. La corrélation de `06` | Part des lignes en promotion et résultat d'un magasin, à mois égal | La promotion fait-elle baisser le résultat, ou est-elle placée là où le résultat est bas ? |
| 10. Verdict | Registre des tests, correction de Bonferroni | Que peut-on conclure ? |
| 11. Le coût | Remises par exercice et par canal, seuil de hausse des ventes qui rendrait une promotion rentable | Que coûtent les promotions, que devraient-elles rapporter ? |
| 12. Les leviers sur l'exercice 2021 | Suppression des remises, par canal et par promotion, grille de réaction des ventes | Que peut-on changer, et pour quel gain ? |
| 13. Synthèse | Réponse à Q2 | |

**Lit dans** : `data/clean/fait_ventes_variables.parquet`, `data/clean/fait_charges.parquet`, `data/clean/agregats/ventes_charges_boutique_mois.parquet`, `data/clean/dimensions`

**Écrit dans** : `reports/figures` (figures `07_*.png`) et `reports/livrables/L4_analyse/q2_tableaux.xlsx`

**Utilise** : `src/analyse.py` (fonctions communes aux notebooks d'analyse : `mann_whitney`, `khi2`, `noter_test`, `tableau_tests`, `eta2`, `exporter_tableaux`)

**Ce que ce notebook ne traite pas** : les **niveaux de remise** (quelle part des unités remiser, à quel taux, quel produit : `09_q5_remises`) et le lien entre promotions et **retours** (`08_q3_q4_retours`).

### Vocabulaire

| Terme | Définition |
|---|---|
| Promotion | campagne saisonnière qui s'applique, pendant sa période, à toutes les lignes d'un groupe de boutiques |
| Taux annoncé | le taux de remise affiché par la promotion (5 %, 7 %, 10 % ou 20 %) |
| Taux de remise effectif | montant de la remise ÷ montant avant remise. C'est ce que la promotion coûte réellement. Il est bien plus bas que le taux annoncé, parce que la remise ne s'applique qu'à une partie des unités de la ligne |
| Ligne remisée | ligne en promotion dont au moins une unité a reçu la remise |
| Ligne | le cumul d'un produit, dans une boutique, un jour donné, pour une promotion. Ce n'est pas un ticket de caisse |
| Nouveauté de l'année | produit vendu l'année de sa mise en vente (`06b_q1_chute_des_ventes`) |
| Ce que rapporte 1 $ de vente | marge nette des retours moins « autres charges », ÷ chiffre d'affaires net : 0,552 $ en 2021, 0,551 $ en magasin (`06_q1_facteurs_rentabilite`, section 8) |
| Hypothèse nulle (H0) | l'hypothèse « il n'y a pas d'effet ». Un test cherche si les données permettent de la rejeter |
| Valeur p | probabilité d'observer un écart au moins aussi grand si H0 était vraie |
| Taille de l'effet | l'ampleur de l'écart (en %, ou sous forme de corrélation). Elle dit si l'écart **compte**, ce que la valeur p ne dit pas |

### Règle de lecture

Avec 715 291 lignes, presque tout écart est « statistiquement significatif ». Chaque test est donc lu sur **deux** critères : la valeur p et la taille de l'effet. Comme le notebook enchaîne plusieurs tests, le seuil de 5 % est divisé par le nombre de tests (**correction de Bonferroni**). Les conclusions reçoivent un verdict : *retenu* (mesuré), *indice* (compatible, sans preuve directe), *non soutenu* (les données ne montrent pas la trace attendue), *ouvert* (les données ne permettent pas de répondre).

## 0. Préparation

In [ ]:
import sys
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# ---------------------------------------------------------------------------
# 1. Retrouver la racine du projet (le dossier qui contient src/ et data/)
# ---------------------------------------------------------------------------
racine = Path.cwd()
while not ((racine / "src").is_dir() and (racine / "data").is_dir()):
    if racine == racine.parent:
        raise FileNotFoundError("Racine du projet introuvable : lancer ce notebook depuis le dossier Projet ADD ou un de ses sous-dossiers.")
    racine = racine.parent
sys.path.insert(0, str(racine))

# ---------------------------------------------------------------------------
# 2. Modules du projet
# ---------------------------------------------------------------------------
#   config  -> chemins, années de l'étude (ANNEES_ETUDE, ANNEE_REFERENCE, EXERCICE)
#   viz     -> charte graphique
#   analyse -> tests (mann_whitney, khi2), registre des tests (noter_test, tableau_tests),
#              eta2 (poids d'un facteur), exporter_tableaux, et les libellés français
from src import config, viz, analyse
from src.analyse import LIBELLES_TYPES, ORDRE_TYPES, NOMS_MOIS, eta2, mann_whitney, khi2, noter_test, tableau_tests

config.creer_dossiers()
viz.appliquer_style()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# ---------------------------------------------------------------------------
# 3. Les trois exercices
# ---------------------------------------------------------------------------
ANNEES = config.ANNEES_ETUDE                  # [2019, 2020, 2021]
REFERENCE, EXERCICE = config.ANNEE_REFERENCE, config.EXERCICE
BASCULE = [a for a in ANNEES if a not in (REFERENCE, EXERCICE)][0]
COULEURS_ANNEES = {REFERENCE: viz.GRIS, BASCULE: viz.CATEGORIES[2], EXERCICE: viz.PRINCIPALE}
print("Racine du projet :", config.ROOT)
print(f"Exercices : {ANNEES} · référence {REFERENCE} · exercice étudié {EXERCICE}")

# Couleur de chaque taux annoncé, la même dans toutes les figures (du plus clair au plus foncé)
COULEURS_TAUX = {0.0: viz.GRILLE, 0.05: viz.SEQUENTIELLE[1], 0.07: viz.SEQUENTIELLE[2],
                 0.10: viz.SEQUENTIELLE[3], 0.20: viz.SEQUENTIELLE[5]}

tableaux = {}      # tableaux à exporter vers Excel à la fin : nom de l'onglet -> tableau
registre = []      # registre des tests d'hypothèse, rempli par noter_test() au fil du notebook
ALPHA = 0.05       # seuil de signification avant correction


def m_dollars(valeur, decimales=1, signe=False):
    """Montant en millions de dollars, virgule française : 12,3 M$."""
    texte = f"{valeur / 1e6:{'+' if signe else ''},.{decimales}f}".replace(",", " ").replace(".", ",")
    return f"{texte} M$"


def pct(valeur, decimales=1, signe=False):
    """Taux en pourcentage, virgule française et espace avant % : 0.123 -> '12,3 %'."""
    return f"{valeur * 100:{'+' if signe else ''}.{decimales}f} %".replace(".", ",")


def nombre(valeur, decimales=0):
    """Nombre avec espace des milliers et virgule décimale : 12345.6 -> '12 345,6'."""
    return f"{valeur:,.{decimales}f}".replace(",", " ").replace(".", ",")


def libelle_taux(t):
    """Libellé d'un taux annoncé : 0.2 -> 'promotion à 20 %', 0 -> 'sans promotion'."""
    return "sans promotion" if t == 0 else f"promotion à {t * 100:.0f} %"

## 1. Chargement

In [ ]:
chemins = {
    "ventes": config.CLEAN / "fait_ventes_variables.parquet",
    "charges": config.CLEAN / "fait_charges.parquet",
    "commun": config.AGREGATS / "ventes_charges_boutique_mois.parquet",
    "dim_boutique": config.CLEAN_DIMENSIONS / "dim_boutique.parquet",
    "dim_produit": config.CLEAN_DIMENSIONS / "dim_produit.parquet",
}
for nom, chemin in chemins.items():
    if not chemin.exists():
        raise FileNotFoundError(f"{chemin} introuvable : exécuter d'abord les notebooks 00 à 05c.")

ventes = pd.read_parquet(chemins["ventes"])
charges = pd.read_parquet(chemins["charges"])
commun = pd.read_parquet(chemins["commun"])
dim_boutique = pd.read_parquet(chemins["dim_boutique"])
dim_produit = pd.read_parquet(chemins["dim_produit"])

# ---------------------------------------------------------------------------
# 1. Libellés communs aux tables
# ---------------------------------------------------------------------------
ventes["type"] = ventes.type_boutique.map(LIBELLES_TYPES)
commun["type"] = commun.type_boutique.map(LIBELLES_TYPES)
reel = charges[charges.scenario == "Réel"].copy()
reel["type"] = reel.type_boutique.map(LIBELLES_TYPES)
ventes["nom_promotion"] = ventes.nom_promotion.str.replace("Scholl", "School")      # coquille de la source

# Magasin ouvert toute l'année : ouvert au 1er janvier de l'année de la vente
ouverture = dim_boutique.set_index("id_boutique").date_ouverture
ventes["annee_complete"] = ventes.id_boutique.map(ouverture) < pd.to_datetime(ventes.annee.astype(str) + "-01-02")

# ---------------------------------------------------------------------------
# 2. Gamme et âge du produit (pour la section 3)
# ---------------------------------------------------------------------------
LIBELLES_CLASSES = {"Economy": "Économique", "Regular": "Standard", "Deluxe": "Luxe"}
ORDRE_GAMMES = ["Économique", "Standard", "Luxe"]
ventes["gamme"] = ventes.classe.map(LIBELLES_CLASSES)
lancement = dim_produit.set_index("id_produit").date_mise_en_vente.dt.year
ventes["annee_de_lancement"] = ventes.id_produit.map(lancement)
ORDRE_AGES = ["nouveauté de l'année", "lancé l'année précédente", "lancé il y a deux ans ou plus"]
ventes["age_produit"] = np.select([ventes.annee <= ventes.annee_de_lancement, ventes.annee - ventes.annee_de_lancement == 1],
                                  ORDRE_AGES[:2], ORDRE_AGES[2])

print(f"Ventes  : {len(ventes):,} lignes · {ventes.id_boutique.nunique()} boutiques · {ventes.id_produit.nunique()} produits".replace(",", " "))
print(f"Période : du {ventes.date.min().date()} au {ventes.date.max().date()}")
nb_promo = int(ventes.a_promotion.sum())
print(f"Lignes en promotion : {nombre(nb_promo)}, soit {pct(nb_promo / len(ventes))} des lignes")
print(f"Lignes corrigées par la règle N6 du nettoyage (quantité remisée ramenée à la quantité vendue) : {int(ventes.remise_corrigee.sum())}")
print("Lignes de promotion hors de la période de la promotion (janvier, 03_diagnostic) :", ventes.groupby("annee").promo_hors_periode.sum().to_dict())

# Garde-fous : trois années complètes, et une table commune qui porte sur les mêmes ventes
if sorted(ventes.annee.unique()) != ANNEES or ventes.groupby("annee").trimestre.nunique().min() != 4:
    raise ValueError(f"La table des ventes ne couvre pas les années {ANNEES} en entier : relancer python -m src.extraction puis les notebooks 00 à 05c.")
if abs(commun.ca_net.sum() - ventes.ca_net.sum()) > 1 or abs(commun.montant_remise.sum() - ventes.montant_remise.sum()) > 1:
    raise ValueError("La table commune ventes et charges ne correspond pas à la table des ventes : relancer 05b_charges.")

## 2. Portrait des promotions

Avant de chercher un effet, on regarde **comment** les promotions sont distribuées : lesquelles, quand, pour qui, et avec quelle remise réelle.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Les promotions présentes dans les ventes, exercice par exercice
# ---------------------------------------------------------------------------
portrait = (ventes.groupby(["annee", "id_promotion", "nom_promotion", "taux_remise_annonce"])
            .agg(lignes=("id_vente", "size"), premiere_vente=("date", "min"), derniere_vente=("date", "max"),
                 boutiques=("id_boutique", "nunique"), produits=("id_produit", "nunique"),
                 montant_avant_remise=("montant_avant_remise", "sum"), montant_remise=("montant_remise", "sum"),
                 part_lignes_remisees=("a_remise", "mean"))
            .reset_index())
portrait["taux_remise_effectif"] = portrait.montant_remise / portrait.montant_avant_remise
portrait = portrait.sort_values(["annee", "premiere_vente", "id_promotion"])
print("Les promotions, exercice par exercice :")
display(portrait.style.format({"taux_remise_annonce": "{:.0%}", "lignes": "{:,.0f}", "premiere_vente": "{:%Y-%m-%d}",
                               "derniere_vente": "{:%Y-%m-%d}", "montant_avant_remise": "{:,.0f}", "montant_remise": "{:,.0f}",
                               "part_lignes_remisees": "{:.1%}", "taux_remise_effectif": "{:.2%}"}).hide(axis="index"))
tableaux["2_promotions"] = portrait.set_index(["annee", "id_promotion"])

# ---------------------------------------------------------------------------
# 2. Qui suit quel calendrier ? Les promotions "North America" ne concernent qu'une boutique
# ---------------------------------------------------------------------------
boutiques_autre_calendrier = ventes.loc[ventes.nom_promotion.str.contains("North America"), "nom_boutique"].unique()
print("Boutique(s) sous promotions « North America » :", list(boutiques_autre_calendrier))
if len(boutiques_autre_calendrier) != 1:
    raise ValueError("Ce notebook suppose qu'une seule boutique suit le calendrier nord-américain.")
NOM_TEMOIN = boutiques_autre_calendrier[0]
ID_TEMOIN = int(ventes.loc[ventes.nom_boutique == NOM_TEMOIN, "id_boutique"].iloc[0])
NOM_COURT = NOM_TEMOIN.replace("Contoso ", "").replace(" Store", "")      # "Cambridge"
autres = ventes[ventes.id_boutique != ID_TEMOIN]                           # toutes les boutiques sauf le témoin


def taux_du_mois(df):
    """Taux annoncé en vigueur chaque mois (le plus fréquent du mois)."""
    return df.groupby("annee_mois").taux_remise_annonce.agg(lambda s: s.mode().iloc[0])


calendrier = pd.DataFrame({"Autres boutiques": taux_du_mois(autres), NOM_COURT: taux_du_mois(ventes[ventes.id_boutique == ID_TEMOIN])})
calendrier["écart"] = calendrier[NOM_COURT] - calendrier["Autres boutiques"]
calendrier["annee"] = calendrier.index.str[:4].astype(int)
calendrier["mois"] = calendrier.index.str[5:7].astype(int)
grille_calendrier = calendrier.pivot_table(index="annee", columns="mois", values=["Autres boutiques", NOM_COURT])
grille_calendrier.columns = [f"{g} · {NOMS_MOIS[m - 1]}" for g, m in grille_calendrier.columns]
meme_calendrier = bool((calendrier.groupby("mois")[["Autres boutiques", NOM_COURT]].nunique() == 1).all().all())
print(f"Taux annoncé en vigueur chaque mois (identique les trois exercices : {'oui' if meme_calendrier else 'non'}) :")
display(calendrier[calendrier.annee == EXERCICE].set_index("mois")[["Autres boutiques", NOM_COURT, "écart"]]
        .rename(index=lambda m: NOMS_MOIS[m - 1]).T.style.format("{:.0%}"))
tableaux["2_calendrier"] = calendrier

# ---------------------------------------------------------------------------
# 3. La promotion est-elle ciblée ? Part des lignes en promotion, par mois, catégorie et canal
# ---------------------------------------------------------------------------
etendue = autres.groupby(["annee_mois", "categorie"]).a_promotion.mean().unstack("categorie")
mois_mixtes = int(((etendue.min(axis=1) > 0.02) & (etendue.max(axis=1) < 0.98)).sum())
mois_promo = int((etendue.min(axis=1) >= 0.98).sum())
mois_sans = int((etendue.max(axis=1) <= 0.02).sum())
print(f"Sur {len(etendue)} mois (toutes boutiques sauf {NOM_COURT}) : {mois_promo} mois où plus de 98 % des lignes de chaque catégorie sont en promotion, "
      f"{mois_sans} mois où moins de 2 % le sont, {mois_mixtes} mois mixtes")
par_canal_mois = autres.groupby(["annee_mois", "type"]).a_promotion.mean().unstack("type")
print(f"Écart maximal entre canaux de la part des lignes en promotion, un mois donné : {pct((par_canal_mois.max(axis=1) - par_canal_mois.min(axis=1)).max())}")

# ---------------------------------------------------------------------------
# 4. Taux annoncé contre taux effectif, sur les lignes en promotion
# ---------------------------------------------------------------------------
en_promo = ventes[ventes.a_promotion].copy()


def remise_par(df, cle):
    r = df.groupby(cle, observed=True).agg(lignes=("id_vente", "size"), part_lignes_remisees=("a_remise", "mean"),
                                           part_unites_remisees=("part_unites_remisees", "mean"),
                                           montant_remise=("montant_remise", "sum"), montant_avant_remise=("montant_avant_remise", "sum"))
    r["taux_remise_effectif"] = r.montant_remise / r.montant_avant_remise
    return r.drop(columns="montant_avant_remise")


FORMATS_REMISE = {"lignes": "{:,.0f}", "part_lignes_remisees": "{:.1%}", "part_unites_remisees": "{:.1%}",
                  "montant_remise": "{:,.0f}", "taux_remise_effectif": "{:.2%}"}
remise_taux = remise_par(en_promo, "taux_remise_annonce")
print("Remise réellement appliquée, par taux annoncé (lignes en promotion, trois exercices) :")
display(remise_taux.style.format(FORMATS_REMISE).format_index("{:.0%}"))
remise_taux_annee = remise_par(en_promo, ["taux_remise_annonce", "annee"]).taux_remise_effectif.unstack("annee")
print("Taux de remise effectif, par taux annoncé et par exercice :")
display(remise_taux_annee.style.format("{:.2%}").format_index("{:.0%}"))
remise_canal = remise_par(en_promo, ["type", "annee"]).loc[ORDRE_TYPES]
print("Remise réellement appliquée, par canal et par exercice (lignes en promotion) :")
display(remise_canal.style.format(FORMATS_REMISE))
tableaux["2_remise_par_taux"] = remise_taux
tableaux["2_remise_par_taux_exercice"] = remise_taux_annee
tableaux["2_remise_par_canal"] = remise_canal

**Lecture**

**Un seul calendrier, le même les trois exercices.**

| Mois | Janv. | Févr. à avr. | Mai à juil. | Août, sept. | Oct. à déc. |
|---|---|---|---|---|---|
| 55 boutiques (calendrier européen) | 20 % (Fêtes de l'année précédente) | 7 % (printemps) | aucune | 10 % (rentrée) | 20 % (Fêtes) |
| Cambridge (calendrier nord-américain) | 5 % | 5 % en févr. et mars, aucune en avril | 10 % en juillet, aucune en mai et juin | 10 % | aucune en oct., 20 % en nov. et déc. |

- **Neuf mois sur douze sont sous promotion**, chaque exercice, et 74,5 % des lignes sont vendues sous promotion.
- **La promotion n'est pas ciblée, elle est générale.** Sur les 36 mois, 27 mois ont plus de 98 % des lignes de chaque catégorie en promotion, 9 mois moins de 2 %, aucun mois ne mélange les deux. La part des lignes en promotion est la même dans les trois canaux, à 0,0 point près, chaque mois.
- **Une seule exception : le magasin de Cambridge**, qui suit le calendrier nord-américain. Six mois par an (janvier à avril, juillet, octobre), il n'est pas sous le même taux que les autres. C'est le seul point de comparaison « même mois, autre promotion » : il sert en section 7.
- On compte six promotions actives en 2019 et sept en 2020 et 2021 : la promotion des Fêtes de l'exercice précédent court encore en janvier. En janvier 2019, ce sont des lignes rattachées à la promotion des Fêtes 2019, hors de sa période (28 454 lignes, `03_diagnostic`), et 1 258 lignes de même nature en janvier 2021 ; elles sont gardées, la remise y a bien été accordée.

**Le taux effectif est très loin du taux annoncé.**

| Taux annoncé | Lignes en promotion | Part des lignes remisées | Taux effectif (trois exercices) | 2019 | 2020 | 2021 |
|---|---|---|---|---|---|---|
| 5 % (Cambridge) | 1 551 | 78,7 % | 1,15 % | 1,31 % | 1,08 % | 1,02 % |
| 7 % | 165 728 | 54,8 % | 0,84 % | 0,85 % | 0,81 % | 0,85 % |
| 10 % | 121 450 | 81,3 % | 2,27 % | 2,44 % | 2,22 % | 2,10 % |
| 20 % | 244 373 | 82,2 % | 5,07 % | 5,46 % | 4,91 % | 4,70 % |

Une promotion « à 20 % » coûte en réalité moins de 5 % du montant de la ligne : la remise ne s'applique qu'à une partie des unités (environ une sur quatre en moyenne), et une partie des lignes n'en reçoit aucune. Toutes les analyses de ce notebook raisonnent donc sur le **taux effectif**. Il est un peu plus élevé en magasin (2,80 % sur les lignes en promotion en 2021) qu'en ligne (2,67 %) et chez le revendeur (2,73 %). Le détail des niveaux de remise est l'objet de `09_q5_remises`.

## 3. L'évolution des promotions de 2019 à 2021, face à l'assortiment

`06b` établit que les ventes ont chuté parce que **l'assortiment vieillit** (552, 429 puis 272 lancements) et que **la clientèle descend en gamme** (la gamme économique passe de 10,6 % à 21,6 % du chiffre d'affaires). Deux questions :

1. **Les promotions ont-elles changé** sur les trois exercices (nombre, taux annoncé, taux effectif, part des lignes, montant) ?
2. **Sur quoi portent-elles** : les nouveautés ou les produits anciens, quelle gamme ? Une promotion qui accompagnerait les nouveautés soutiendrait le renouvellement de l'assortiment ; une promotion qui porterait surtout sur l'entrée de gamme accompagnerait la descente en gamme.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Les promotions par exercice
# ---------------------------------------------------------------------------
def indicateurs_promotion(df):
    """Indicateurs de promotion d'un ensemble de lignes (toutes lignes, en promotion ou non)."""
    p = df[df.a_promotion]
    return pd.Series({
        "promotions actives": p.id_promotion.nunique(),
        "mois sous promotion": int((df.groupby("annee_mois").a_promotion.mean() > 0.5).sum()),
        "part des lignes en promotion": df.a_promotion.mean(),
        "taux annoncé moyen": (p.taux_remise_annonce * p.montant_avant_remise).sum() / p.montant_avant_remise.sum(),
        "taux effectif des lignes en promotion": p.montant_remise.sum() / p.montant_avant_remise.sum(),
        "part des lignes en promotion qui sont remisées": p.a_remise.mean(),
        "part des unités remisées (ligne remisée)": p[p.a_remise].part_unites_remisees.mean(),
        "remises": df.montant_remise.sum(),
        "chiffre d'affaires net": df.ca_net.sum(),
        "remises ÷ CA net": df.montant_remise.sum() / df.ca_net.sum(),
    })


evolution = pd.DataFrame({a: indicateurs_promotion(ventes[ventes.annee == a]) for a in ANNEES})
for a, b in [(REFERENCE, BASCULE), (BASCULE, EXERCICE), (REFERENCE, EXERCICE)]:
    evolution[f"{a} → {b}"] = evolution[b] / evolution[a] - 1
FORMATS_EVOLUTION = {"promotions actives": "{:.0f}", "mois sous promotion": "{:.0f}", "remises": "{:,.0f}", "chiffre d'affaires net": "{:,.0f}"}


def montrer_evolution(table):
    """Affiche le tableau d'évolution : une mise en forme par ligne (nombre, montant ou taux) et des évolutions signées."""
    texte = table.copy().astype(object)
    for ligne in table.index:
        for col in table.columns:
            v = table.loc[ligne, col]
            if pd.isna(v):
                texte.loc[ligne, col] = ""
            elif "→" in str(col):
                texte.loc[ligne, col] = f"{v:+.1%}"
            else:
                texte.loc[ligne, col] = FORMATS_EVOLUTION.get(ligne, "{:.2%}").format(v)
    display(texte)


print("Les promotions par exercice, toutes boutiques :")
montrer_evolution(evolution)
tableaux["3_evolution"] = evolution

# Par canal et par exercice
evolution_canal = pd.concat({t: pd.DataFrame({a: indicateurs_promotion(ventes[(ventes.annee == a) & (ventes.type == t)]) for a in ANNEES})
                             for t in ORDRE_TYPES}, names=["canal", "indicateur"])
choix = ["part des lignes en promotion", "taux effectif des lignes en promotion", "remises", "remises ÷ CA net"]
canal_large = evolution_canal.loc[(slice(None), choix), :]
canal_large[f"{REFERENCE} → {EXERCICE}"] = canal_large[EXERCICE] / canal_large[REFERENCE] - 1
print("Par canal :")
display(canal_large.style.format(lambda v: f"{v:,.0f}" if abs(v) > 10 else f"{v:.2%}"))
tableaux["3_evolution_canal"] = evolution_canal

# Unités par ligne et unités remisées par ligne remisée : ce qui fait bouger le taux effectif
lignes_remisees = ventes[ventes.a_remise]
unites = pd.DataFrame({"unités par ligne remisée": lignes_remisees.groupby("annee").quantite_vente.mean(),
                       "unités remisées par ligne remisée": lignes_remisees.groupby("annee").quantite_remise.mean()})
print("Lignes remisées : unités vendues et unités remisées par ligne, par exercice :")
display(unites.T.style.format("{:.2f}"))
tableaux["3_unites_remisees"] = unites

# ---------------------------------------------------------------------------
# 2. Sur quoi portent les promotions : âge du produit et gamme
# ---------------------------------------------------------------------------
def cible(cle, ordre):
    """Par exercice et par groupe : part des lignes en promotion, taux effectif, part des remises et part du CA."""
    g = ventes.groupby(["annee", cle]).agg(lignes=("id_vente", "size"), part_lignes_en_promotion=("a_promotion", "mean"),
                                           remises=("montant_remise", "sum"), avant_remise=("montant_avant_remise", "sum"),
                                           ca_net=("ca_net", "sum"))
    g["taux_effectif"] = g.remises / g.avant_remise
    g["part_des_remises"] = g.remises / g.groupby("annee").remises.transform("sum")
    g["part_du_ca"] = g.ca_net / g.groupby("annee").ca_net.transform("sum")
    g["remises_par_dollar_de_ca"] = g.remises / g.ca_net
    return g.drop(columns="avant_remise").reindex(pd.MultiIndex.from_product([ANNEES, ordre], names=["annee", cle]))


FORMATS_CIBLE = {"lignes": "{:,.0f}", "part_lignes_en_promotion": "{:.1%}", "remises": "{:,.0f}", "ca_net": "{:,.0f}",
                 "taux_effectif": "{:.2%}", "part_des_remises": "{:.1%}", "part_du_ca": "{:.1%}", "remises_par_dollar_de_ca": "{:.3f}"}
cible_age = cible("age_produit", ORDRE_AGES)
cible_gamme = cible("gamme", ORDRE_GAMMES)
print("Promotions selon l'âge du produit :")
display(cible_age.style.format(FORMATS_CIBLE))
print("Promotions selon la gamme :")
display(cible_gamme.style.format(FORMATS_CIBLE))
tableaux["3_promotions_age"] = cible_age
tableaux["3_promotions_gamme"] = cible_gamme

# Tests : la part des lignes en promotion dépend-elle de l'âge du produit, de la gamme ? (khi-deux et V de Cramér)
for cle, libelle in [("age_produit", "l'âge du produit"), ("gamme", "la gamme")]:
    v, p = khi2(ventes.a_promotion, ventes[cle])
    noter_test(registre, f"Ciblage : la part des lignes en promotion dépend-elle de {libelle} ?",
               f"Même part des lignes en promotion quel que soit {libelle}", "Khi-deux", len(ventes), f"V de Cramér = {v:.3f}", p)

# Les nouveautés lancées en cours d'année : leurs lignes en promotion selon le mois de lancement
nouveautes = ventes[ventes.age_produit == ORDRE_AGES[0]]
mois_lancement = nouveautes.id_produit.map(dim_produit.set_index("id_produit").date_mise_en_vente.dt.month)
lancement_promo = nouveautes.groupby(mois_lancement).agg(lignes=("id_vente", "size"), part_lignes_en_promotion=("a_promotion", "mean"))
lancement_promo.index = [NOMS_MOIS[int(m) - 1] for m in lancement_promo.index]
print("Nouveautés de l'année, selon leur mois de lancement (trois exercices) :")
display(lancement_promo.T.style.format(lambda v: f"{v:,.0f}" if v > 1 else f"{v:.0%}"))
tableaux["3_nouveautes_mois_lancement"] = lancement_promo

# ---------------------------------------------------------------------------
# 3. Graphique : la part des lignes en promotion ne dépend ni de l'âge ni de la gamme
# ---------------------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.4), sharey=True)
for ax, table, ordre, nom in [(ax1, cible_age, ORDRE_AGES, "âge du produit"), (ax2, cible_gamme, ORDRE_GAMMES, "gamme")]:
    largeur = 0.26
    for i, a in enumerate(ANNEES):
        valeurs = table.loc[a, "part_lignes_en_promotion"].reindex(ordre)
        x = np.arange(len(ordre)) + (i - 1) * largeur
        ax.bar(x, valeurs, width=largeur, color=COULEURS_ANNEES[a], label=str(a))
        for xi, v in zip(x, valeurs):
            ax.text(xi, v + 0.01, f"{v * 100:.0f}", ha="center", fontsize=8, color=viz.FONCE)
    ax.set_xticks(range(len(ordre)), [textwrap.fill(o, 16) for o in ordre])
    ax.set_ylim(0, 1.05)
    viz.format_pourcent(ax, axe="y")
    viz.titre(ax, f"Selon l'{nom}" if nom.startswith("â") else f"Selon la {nom}", "Part des lignes en promotion, par exercice")
ax1.legend(frameon=False, loc="upper left", ncol=3)
fig.suptitle("Les promotions touchent les nouveautés comme les produits anciens, toutes les gammes", x=0.02, ha="left",
             fontsize=13, fontweight="bold", color=viz.FONCE, y=1.04)
print("Figure enregistrée :", viz.enregistrer(fig, "07_cible_des_promotions"))
plt.show()

**Lecture**

**Les promotions n'ont pas changé : même calendrier, même taux annoncé, même étendue.**

| Toutes boutiques | 2019 | 2020 | 2021 | 2019 → 2021 |
|---|---|---|---|---|
| Promotions actives | 6 | 7 | 7 | la promotion des Fêtes court sur janvier |
| Mois sous promotion | 9 | 9 | 9 | inchangé |
| Part des lignes en promotion | 75,2 % | 74,0 % | 73,9 % | −1,3 point |
| Taux annoncé moyen | 12,8 % | 12,7 % | 13,1 % | stable |
| Taux effectif des lignes en promotion | 3,04 % | 2,72 % | 2,73 % | −10,1 % |
| Remises | 22,1 M$ | 15,5 M$ | 13,9 M$ | −37,1 % |
| Remises ÷ CA net | 2,33 % | 2,05 % | 2,02 % | −0,3 point |

- **Le seul changement est le taux effectif**, qui baisse de 10 % sans que la direction ait changé de taux annoncé. Il vient du panier : une ligne remisée compte 10,4 unités en 2019, 14,6 en 2020 et 20,7 en 2021, mais toujours environ 2,9 unités remisées (2,86, 2,91, 2,87). La part des unités remisées dans une ligne remisée tombe donc de 39,5 % à 23,2 %. Pourquoi le nombre d'unités remisées ne suit pas la taille du panier relève de `09_q5_remises`.
- **Les remises baissent de 37,1 %, plus vite que le chiffre d'affaires (−27,2 %)**, surtout en magasin (12,4 → 6,4 → 5,5 M$, −55,8 %), où les ventes ont chuté. En ligne, elles reculent de 6,3 % (5,6 → 5,3 M$) ; chez le revendeur, de 22,0 % (4,0 → 3,1 M$). Dans les trois canaux, elles pèsent environ 2 % du chiffre d'affaires en 2021 (magasins 2,08 %, en ligne 1,96 %, revendeur 2,01 %).

**Les promotions portent sur tout l'assortiment, sans choix entre nouveautés et produits anciens.**

| Part des lignes en promotion | 2019 | 2020 | 2021 |
|---|---|---|---|
| Nouveautés de l'année | 73,0 % | 74,3 % | 71,2 % |
| Lancés l'année précédente | 76,0 % | 73,9 % | 73,9 % |
| Lancés il y a deux ans ou plus | 75,7 % | 73,9 % | 74,3 % |
| Gamme économique | 75,4 % | 74,0 % | 74,0 % |
| Gamme standard | 74,6 % | 73,8 % | 74,2 % |
| Gamme luxe | 76,8 % | 74,4 % | 72,9 % |

- Les écarts sont de quelques points ; les tests du khi-deux les jugent « significatifs » (715 291 lignes), mais l'effet est négligeable (V de Cramér 0,016 pour l'âge, 0,007 pour la gamme, quand 0,1 marque un petit effet).
- Le seul écart visible vient du calendrier : une nouveauté lancée en août ou après est vendue presque entièrement sous promotion (99 % à 100 % de ses lignes), une nouveauté lancée en avril ou en mai l'est à 65 % ou 66 %, parce qu'elle tombe en partie dans les mois sans promotion. Ce n'est pas un ciblage.
- **Les remises suivent donc le chiffre d'affaires.** En 2021, les produits lancés il y a deux ans ou plus reçoivent 67,6 % des remises pour 70,8 % du chiffre d'affaires ; les nouveautés de l'année, 12,9 % des remises pour 11,8 % du chiffre d'affaires. En 2019, les nouveautés recevaient 38,1 % des remises : leur part a baissé parce qu'il y a moins de nouveautés (`06b`), pas parce que la promotion s'en est détournée.
- **Par gamme, un glissement en sens contraire de la descente en gamme.** La gamme économique passe de 10,6 % à 21,6 % du chiffre d'affaires, mais de 18,0 % à 10,4 % des remises : son taux effectif tombe de 3,71 % à 0,96 %. Une ligne économique compte de plus en plus d'unités, avec toujours environ trois unités remisées (`09`). Le luxe reçoit 31,8 % des remises en 2021 pour 26,4 % du chiffre d'affaires.

**Ce qu'on en retient.** La promotion n'a été ni un frein, ni un accompagnement de la descente en gamme et du vieillissement de l'assortiment : elle s'applique à tout, de la même façon, chaque exercice. Elle n'est pas utilisée pour lancer les nouveautés ni pour défendre une gamme. *Interprétation métier (hypothèse)* : c'est un calendrier commercial reconduit d'un exercice à l'autre, comme la publicité et les frais généraux (`06`), sans lien avec l'état de l'assortiment.

## 4. Le piège : promotion et saison sont confondues

Le graphique ci-dessous superpose le chiffre d'affaires mensuel et la promotion en vigueur, sur 36 mois. Il montre le problème central de ce notebook : **la couleur change exactement quand le mois change**, et de la même façon chaque année. Savoir qu'on est en promotion à 20 %, c'est savoir qu'on est entre octobre et janvier.

In [ ]:
mensuel = autres.groupby("annee_mois").agg(ca_net=("ca_net", "sum"))
mensuel["taux"] = calendrier["Autres boutiques"]

fig, ax = plt.subplots(figsize=(12, 4.6))
positions = np.arange(len(mensuel))
ax.bar(positions, mensuel.ca_net, color=[COULEURS_TAUX[t] for t in mensuel.taux], width=0.78, edgecolor=viz.GRIS, linewidth=0.4)
ax.set_xticks(positions[::2], [str(m) for m in mensuel.index[::2]], rotation=45, ha="right")
viz.format_dollars(ax, axe="y")
for taux in sorted(mensuel.taux.unique()):
    ax.bar([0], [0], color=COULEURS_TAUX[taux], edgecolor=viz.GRIS, linewidth=0.4, label=libelle_taux(taux).capitalize())
ax.legend(ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.22), frameon=False)
viz.titre(ax, "La promotion change avec le mois, de la même façon chaque année : les deux sont confondues",
          f"Chiffre d'affaires net mensuel, coloré selon la promotion en vigueur · toutes boutiques sauf {NOM_COURT}")
print("Figure enregistrée :", viz.enregistrer(fig, "07_calendrier_promotions"))
plt.show()

# Indice de saisonnalité (100 = mois moyen de l'année) selon la promotion du mois
mensuel["annee"] = mensuel.index.str[:4].astype(int)
mensuel["indice"] = mensuel.ca_net / mensuel.groupby("annee").ca_net.transform("mean") * 100
indice_taux = mensuel.pivot_table(index="taux", columns="annee", values="indice", aggfunc="mean")
indice_taux.index = [libelle_taux(t) for t in indice_taux.index]
print("Indice de saisonnalité moyen des mois, selon la promotion en vigueur :")
display(indice_taux.style.format("{:.0f}"))
tableaux["4_indice_par_promotion"] = indice_taux

**Conséquence pour la méthode**

Pour mesurer l'effet d'une promotion, il faut un **groupe témoin** : des produits ou des magasins comparables qui, au même moment, n'en ont pas. Ici, il n'y en a presque pas : un mois donné, tous les produits de toutes les boutiques européennes sont sous la même promotion. Une comparaison « avec ou sans promotion » compare donc en réalité mai à juillet au reste de l'année.

On procède en deux temps :

1. **la comparaison naïve** (section 5), pour montrer ce qu'elle donne et pourquoi on ne peut pas s'y fier ;
2. **trois angles qui contournent le problème**, chacun avec sa faiblesse (sections 6 à 8). Aucun n'est une preuve à lui seul : c'est leur **accord** qui fera la conclusion.

| Angle | Idée | Pourquoi la saison gêne moins | Faiblesse |
|---|---|---|---|
| A. Changements de promotion | Comparer les 14 jours avant et après chaque changement | La saison ne change pas d'un coup en 28 jours | Si autre chose change le 1er du mois, on ne le distingue pas |
| B. Magasin de Cambridge | Il suit un autre calendrier : le comparer aux autres, le même mois | Même mois, promotion différente | Un seul magasin |
| C. Lignes remisées | Lignes remisées et non remisées du même produit, le même mois | Même produit, même mois | Lien en partie mécanique (section 8) |

## 5. La comparaison naïve (biaisée)

On compare les lignes selon le taux annoncé, tous mois confondus.

- **H0** : la taille d'une ligne (unités vendues) est la même quel que soit le taux annoncé.
- **Tests** : Kruskal-Wallis pour les cinq groupes à la fois, puis Mann-Whitney entre « en promotion » et « sans promotion », au niveau de la ligne puis au niveau du jour. Ce sont des tests sur les **rangs** : ils ne supposent pas que les quantités suivent une loi normale, ce qui n'est pas le cas.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Indicateurs par taux annoncé, trois exercices réunis
# ---------------------------------------------------------------------------
def selon_le_taux(df):
    r = df.groupby("taux_remise_annonce").agg(lignes=("id_vente", "size"), jours=("date", "nunique"), unites=("quantite_vente", "sum"),
                                              ca_net=("ca_net", "sum"), marge_nette=("marge_nette", "sum"),
                                              mediane_unites_par_ligne=("quantite_vente", "median"))
    r["lignes_par_jour"] = r.lignes / r.jours
    r["unites_par_ligne"] = r.unites / r.lignes
    r["ca_par_ligne"] = r.ca_net / r.lignes
    r["ca_par_jour"] = r.ca_net / r.jours
    r["taux_de_marge"] = r.marge_nette / r.ca_net
    return r[["lignes", "jours", "lignes_par_jour", "unites_par_ligne", "mediane_unites_par_ligne", "ca_par_ligne", "ca_par_jour", "taux_de_marge"]]


FORMATS_NAIF = {"lignes": "{:,.0f}", "jours": "{:,.0f}", "lignes_par_jour": "{:,.0f}", "unites_par_ligne": "{:.1f}",
                "mediane_unites_par_ligne": "{:.0f}", "ca_par_ligne": "{:,.0f}", "ca_par_jour": "{:,.0f}", "taux_de_marge": "{:.1%}"}
naif = selon_le_taux(autres)
print(f"Ventes selon le taux annoncé, toutes boutiques sauf {NOM_COURT}, trois exercices :")
display(naif.style.format(FORMATS_NAIF).format_index("{:.0%}"))
tableaux["5_comparaison_naive"] = naif

# Par exercice : chiffre d'affaires par jour, rapporté aux jours sans promotion de l'année
naif_annee = pd.concat({a: selon_le_taux(autres[autres.annee == a]) for a in ANNEES}, names=["annee"])
ca_jour = naif_annee.ca_par_jour.unstack("annee")
ecart_jour = ca_jour / ca_jour.loc[0.0] - 1
print("Chiffre d'affaires par jour, écart aux jours sans promotion de l'exercice :")
display(ecart_jour.style.format("{:+.1%}").format_index("{:.0%}"))
naif_canal = pd.concat({t: selon_le_taux(autres[autres.type == t]).ca_par_jour for t in ORDRE_TYPES}, axis=1)
print("Chiffre d'affaires par jour, écart aux jours sans promotion, par canal (trois exercices) :")
display((naif_canal / naif_canal.loc[0.0] - 1).style.format("{:+.1%}").format_index("{:.0%}"))
tableaux["5_naif_par_exercice"] = naif_annee
tableaux["5_naif_par_canal"] = naif_canal

# ---------------------------------------------------------------------------
# 2. Tests
# ---------------------------------------------------------------------------
groupes = [g.quantite_vente.values for _, g in autres.groupby("taux_remise_annonce")]
h, p_kw = stats.kruskal(*groupes)
epsilon2 = h / (len(autres) - 1)          # taille d'effet du Kruskal-Wallis, se lit comme η²
noter_test(registre, "Naïf : la taille d'une ligne dépend-elle du taux annoncé ?", "Même quantité par ligne pour tous les taux",
           "Kruskal-Wallis", len(autres), f"ε² = {epsilon2:.3f}", p_kw)

effet, p_mw = mann_whitney(autres[autres.a_promotion].quantite_vente, autres[~autres.a_promotion].quantite_vente)
noter_test(registre, "Naïf : une ligne en promotion est-elle plus grosse qu'une ligne sans promotion ?",
           "Même quantité par ligne avec et sans promotion", "Mann-Whitney", len(autres), f"corrélation de rang = {effet:+.3f}", p_mw)

# Au niveau du jour, dans chaque exercice : chiffre d'affaires quotidien rapporté à la moyenne de l'exercice
par_jour = autres.groupby("date").agg(ca_net=("ca_net", "sum"), promo=("a_promotion", "mean"), annee=("annee", "first"))
par_jour["promo"] = par_jour.promo > 0.5
par_jour["ca_relatif"] = par_jour.ca_net / par_jour.groupby("annee").ca_net.transform("mean")
effet_j, p_j = mann_whitney(par_jour.ca_relatif[par_jour.promo], par_jour.ca_relatif[~par_jour.promo])
ecart_j = par_jour.ca_relatif[par_jour.promo].median() / par_jour.ca_relatif[~par_jour.promo].median() - 1
noter_test(registre, "Naïf : vend-on plus les jours de promotion (à exercice égal) ?", "Même chiffre d'affaires quotidien avec et sans promotion",
           "Mann-Whitney", len(par_jour), f"écart des médianes = {pct(ecart_j, signe=True)} · corrélation de rang = {effet_j:+.2f}", p_j)

**Lecture**

**Le résultat brut va à l'inverse de l'intuition.**

| Toutes boutiques sauf Cambridge, trois exercices | Sans promotion | 7 % | 10 % | 20 % |
|---|---|---|---|---|
| Lignes par jour | 650 | 618 | 655 | 659 |
| Unités par ligne | 15,3 | 17,2 | 13,6 | 12,1 |
| Chiffre d'affaires par jour | 2,38 M$ | 2,45 M$ | 2,12 M$ | 1,82 M$ |
| Taux de marge | 57,1 % | 56,1 % | 56,5 % | 55,1 % |

- Plus le taux annoncé est élevé, plus les lignes sont **petites** et plus le chiffre d'affaires par jour est **bas** : −23,6 % les jours de promotion à 20 % par rapport aux jours sans promotion. Le nombre de lignes par jour, lui, ne change presque pas (618 à 659).
- **Le constat tient les trois exercices et dans les trois canaux** : à 20 %, le chiffre d'affaires par jour est de 19,6 %, 25,8 % et 26,4 % inférieur à celui des jours sans promotion de l'exercice ; −27,4 % en ligne, −25,1 % chez le revendeur, −19,9 % en magasin. La promotion à 7 % (février à avril) est la seule proche des jours sans promotion (+11,9 % en 2019, +5,3 % en 2020, −10,0 % en 2021).
- **Les trois tests rejettent H0**, avec des effets non négligeables : ε² = 0,126 pour le taux annoncé, corrélation de rang −0,168 entre lignes avec et sans promotion, et au niveau du jour, à exercice égal, un jour de promotion vend 12,9 % de moins en médiane qu'un jour sans promotion (corrélation de rang −0,60).
- **Mais on ne peut rien en conclure sur l'effet des promotions.** Ces tests comparent des périodes différentes de l'année. Ils établissent que les mois de promotion sont des mois de ventes plus faibles (indice de saisonnalité 83 à 86 pour la promotion à 20 %, 106 à 116 pour les mois sans promotion, section 4). Ils ne disent pas si c'est **à cause** de la promotion, ou parce que la promotion est placée sur des mois faibles.
- Un seul fait solide : **rien, dans les chiffres bruts, ne montre que les promotions font vendre davantage.**

## 6. Angle A : que se passe-t-il quand la promotion change ?

**L'idée.** La promotion change quatre fois par an, toujours un 1er du mois. Si elle a un effet sur les ventes, il doit se voir **à ce moment précis** : on compare, pour chaque boutique, ses ventes des 14 jours qui précèdent le changement à celles des 14 jours qui suivent. En 28 jours, la saison n'a pas le temps de beaucoup bouger.

**Le contrôle.** On fait le même calcul pour **tous les débuts de mois**, y compris ceux où la promotion ne change pas. Ces dates servent de **placebo** : si les ventes sautaient à chaque 1er du mois, promotion ou pas, le saut ne viendrait pas de la promotion. Les deux 1er janvier (2020 et 2021) sont mis à part : ils changent d'exercice, et les ventes changent de niveau d'un exercice à l'autre (`06`).

- **H0** (pour chaque date) : les ventes d'une boutique sont les mêmes avant et après la date. Test de Wilcoxon apparié (chaque boutique est comparée à elle-même).
- **Puis** : le saut est-il lié au sens du changement de taux ? Corrélation de Spearman entre la variation du taux et la variation des ventes, sur l'ensemble des dates.

On dispose de 35 dates (du 1er février 2019 au 1er décembre 2021), dont 12 changements de taux (quatre par exercice), 21 dates sans changement de taux et deux 1er janvier.

In [ ]:
FENETRE = 14      # nombre de jours comparés de chaque côté de la date

# Ventes par boutique et par jour, et taux annoncé en vigueur chaque jour
boutique_jour = autres.groupby(["id_boutique", "date"]).agg(ca_net=("ca_net", "sum"), unites=("quantite_vente", "sum"),
                                                           lignes=("id_vente", "size")).reset_index()
type_de = autres.groupby("id_boutique").type.first()
taux_du_jour = autres.groupby("date").taux_remise_annonce.agg(lambda s: s.mode().iloc[0])
DATES = pd.date_range(f"{REFERENCE}-02-01", f"{EXERCICE}-12-01", freq="MS")


def variation_autour(date):
    """Logarithme du rapport (14 jours après ÷ 14 jours avant), boutique par boutique."""
    debut, fin = date - pd.Timedelta(days=FENETRE), date + pd.Timedelta(days=FENETRE)
    avant = boutique_jour[(boutique_jour.date >= debut) & (boutique_jour.date < date)].groupby("id_boutique")[["ca_net", "unites", "lignes"]].sum()
    apres = boutique_jour[(boutique_jour.date >= date) & (boutique_jour.date < fin)].groupby("id_boutique")[["ca_net", "unites", "lignes"]].sum()
    return np.log(apres / avant).replace([np.inf, -np.inf], np.nan).dropna()     # les boutiques absentes d'une fenêtre sortent


resultats, variations = [], {}
for date in DATES:
    variation = variation_autour(date)
    variations[date] = variation.ca_net
    _, p = stats.wilcoxon(variation.ca_net)
    magasins_v = variation.ca_net[variation.index.map(type_de) == "Magasins"]
    ligne = {"date": date, "taux_avant": taux_du_jour[date - pd.Timedelta(days=1)], "taux_apres": taux_du_jour[date],
             "boutiques": len(variation), "variation_ca": np.exp(variation.ca_net.median()) - 1,
             "variation_unites": np.exp(variation.unites.median()) - 1, "variation_lignes": np.exp(variation.lignes.median()) - 1,
             "part_boutiques_en_hausse": (variation.ca_net > 0).mean(), "p": p,
             "magasins (médiane)": np.exp(magasins_v.median()) - 1}
    for t in ["En ligne", "Revendeur"]:
        v_t = variation.ca_net[variation.index.map(type_de) == t]
        ligne[t] = np.exp(v_t.iloc[0]) - 1 if len(v_t) else np.nan
    resultats.append(ligne)
bascules = pd.DataFrame(resultats).set_index("date")
bascules["variation_taux"] = bascules.taux_apres - bascules.taux_avant
bascules["nature"] = np.select([bascules.index.month == 1, bascules.variation_taux > 0, bascules.variation_taux < 0],
                               ["changement d'exercice", "le taux monte", "le taux baisse"], "taux inchangé (placebo)")
print(f"Ventes des {FENETRE} jours après chaque début de mois, comparées aux {FENETRE} jours avant (médiane des boutiques) :")
display(bascules.style.format({"taux_avant": "{:.0%}", "taux_apres": "{:.0%}", "variation_taux": "{:+.0%}", "variation_ca": "{:+.1%}",
                               "variation_unites": "{:+.1%}", "variation_lignes": "{:+.1%}", "part_boutiques_en_hausse": "{:.0%}",
                               "p": "{:.2g}", "magasins (médiane)": "{:+.1%}", "En ligne": "{:+.1%}", "Revendeur": "{:+.1%}"})
        .format_index("{:%Y-%m-%d}"))
tableaux["6_bascules"] = bascules

# ---------------------------------------------------------------------------
# 2. Synthèse par nature de date et par canal
# ---------------------------------------------------------------------------
synthese_a = bascules.groupby("nature").agg(dates=("variation_ca", "size"), variation_ca_moyenne=("variation_ca", "mean"),
                                            variation_ca_min=("variation_ca", "min"), variation_ca_max=("variation_ca", "max"),
                                            magasins=("magasins (médiane)", "mean"), en_ligne=("En ligne", "mean"),
                                            revendeur=("Revendeur", "mean"), variation_lignes_moyenne=("variation_lignes", "mean"))
print("Moyenne des variations, par nature de date et par canal :")
display(synthese_a.style.format({c: "{:+.1%}" for c in synthese_a.columns if c != "dates"}))
tableaux["6_bascules_synthese"] = synthese_a
par_transition = bascules[bascules.variation_taux != 0].copy()
par_transition["transition"] = [f"{libelle_taux(a)} → {libelle_taux(b)}" for a, b in zip(par_transition.taux_avant, par_transition.taux_apres)]
par_transition["mois"] = par_transition.index.month
transitions = par_transition.pivot_table(index=["mois", "transition"], columns=par_transition.index.year, values="variation_ca")
print("Variation du chiffre d'affaires à chaque changement de taux, par exercice :")
display(transitions.style.format("{:+.1%}", na_rep=""))
tableaux["6_transitions"] = transitions

# ---------------------------------------------------------------------------
# 3. Tests
# ---------------------------------------------------------------------------
hors_exercice = bascules[bascules.nature != "changement d'exercice"]
rho, p_rho = stats.spearmanr(hors_exercice.variation_taux, hors_exercice.variation_ca)
noter_test(registre, "Angle A : quand le taux de promotion monte, les ventes montent-elles ?",
           "Aucun lien entre la variation du taux et la variation des ventes", "Spearman", len(hors_exercice), f"ρ = {rho:+.2f}", p_rho)
for nature, question in [("le taux monte", "Angle A : les ventes changent-elles quand le taux monte ?"),
                         ("le taux baisse", "Angle A : les ventes changent-elles quand le taux baisse ?"),
                         ("taux inchangé (placebo)", "Angle A, placebo : les ventes changent-elles un 1er du mois sans changement de taux ?")]:
    toutes = pd.concat([variations[d] for d in bascules.index[bascules.nature == nature]])
    _, p = stats.wilcoxon(toutes)
    noter_test(registre, question, "Mêmes ventes avant et après la date", "Wilcoxon apparié", len(toutes),
               f"variation médiane = {pct(np.exp(toutes.median()) - 1, signe=True)}", p)
# Vérification par exercice du lien d'ensemble
for a in ANNEES:
    sous = hors_exercice[hors_exercice.index.year == a]
    r_a, p_a = stats.spearmanr(sous.variation_taux, sous.variation_ca)
    print(f"Exercice {a} : ρ = {r_a:+.2f} sur {len(sous)} dates (p = {p_a:.2g})".replace(".", ","))

# ---------------------------------------------------------------------------
# 4. Graphique : la variation des ventes à chaque début de mois
# ---------------------------------------------------------------------------
COULEURS_NATURE = {"le taux monte": viz.ALERTE, "le taux baisse": viz.PRINCIPALE,
                   "taux inchangé (placebo)": viz.GRILLE, "changement d'exercice": viz.ACCENT}
fig, ax = plt.subplots(figsize=(12.5, 4.8))
positions = np.arange(len(bascules))
ax.bar(positions, bascules.variation_ca, color=[COULEURS_NATURE[n] for n in bascules.nature], edgecolor=viz.GRIS, linewidth=0.5)
ax.axhline(0, color=viz.GRIS, linewidth=1)
etiquettes = [f"{d:%Y-%m} · {a * 100:.0f} → {b * 100:.0f}" for d, a, b in zip(bascules.index, bascules.taux_avant, bascules.taux_apres)]
ax.set_xticks(positions, etiquettes, fontsize=7.5, rotation=60, ha="right")
viz.format_pourcent(ax, axe="y")
for nature, couleur in COULEURS_NATURE.items():
    ax.bar([0], [0], color=couleur, edgecolor=viz.GRIS, linewidth=0.5, label=nature.capitalize())
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.36), frameon=False)
viz.titre(ax, "Les ventes reculent quand le taux monte, repartent quand il baisse",
          f"Variation du chiffre d'affaires entre les {FENETRE} jours avant et après chaque 1er du mois · médiane des boutiques · sous chaque date : taux avant → après (%)")
print("Figure enregistrée :", viz.enregistrer(fig, "07_bascules"))
plt.show()

**Lecture**

**Quand le taux de promotion monte, les ventes reculent, aux six dates et dans les trois canaux.**

| Changement de taux | 2019 | 2020 | 2021 |
|---|---|---|---|
| 1er août : sans promotion → 10 % | −9,6 % | −11,6 % | −10,0 % |
| 1er octobre : 10 % → 20 % | −9,1 % | −8,9 % | −16,6 % |
| 1er février : 20 % → 7 % | +62,5 % | +75,1 % | +34,8 % |
| 1er mai : 7 % → sans promotion | −24,8 % | −4,0 % | −1,0 % |

| Nature de la date | Dates | Variation moyenne | Magasins | En ligne | Revendeur |
|---|---|---|---|---|---|
| Le taux monte | 6 | −10,9 % | −10,3 % | −13,8 % | −12,0 % |
| Le taux baisse | 6 | +23,8 % | +23,3 % | +30,3 % | +30,4 % |
| Taux inchangé (placebo) | 21 | +0,1 % | +0,1 % | −0,5 % | +0,5 % |
| Changement d'exercice (1er janvier) | 2 | −44,2 % | −44,6 % | −14,7 % | −23,6 % |

- **Quand le taux monte**, la baisse médiane d'une boutique est de **−9,9 %** (323 observations boutique × date). Elle passe par la taille des lignes, pas par leur nombre (+0,2 % de lignes en moyenne).
- **Quand le taux baisse, les ventes repartent**, mais surtout au 1er février (fin de la promotion à 20 %) ; au 1er mai (fin de la promotion à 7 %), le mouvement est fort en 2019 seulement, et dans le mauvais sens (−24,8 %). La hausse médiane est de **+23,5 %**.
- **Le placebo confirme que la date ne suffit pas** : les 1er du mois sans changement de taux, les ventes ne bougent pas en médiane (+0,2 %, p = 0,78). Deux dates placebo s'écartent toutefois les trois années : le 1er mars (−8,7 %, −14,5 %, −14,0 %) et, en 2019 et 2021, le 1er avril (+13,1 %, +12,3 %). Tout n'est pas expliqué par la promotion. Les deux 1er janvier marquent le changement de niveau entre exercices (−59,2 % en 2020, −29,2 % en 2021, porté par le nombre de lignes, `06`).
- **Le lien d'ensemble est négatif** : plus le taux monte, plus les ventes baissent (ρ = −0,50 sur 33 dates, p = 0,0029, sous le seuil corrigé). Il se retrouve dans chaque exercice (−0,39, −0,55, −0,60), sans être significatif sur onze dates seulement.
- **Ce que cet angle établit** : les ventes changent au moment où la promotion change, **dans le mauvais sens**. Une promotion qui ferait vendre montrerait l'inverse. **Sa limite** : si autre chose change chaque année le même jour que la promotion (une saison commerciale qui commence le 1er août ou le 1er octobre, par exemple), on ne peut pas le distinguer avec cet angle seul. C'est l'objet de l'angle B.

## 7. Angle B : le magasin de Cambridge

**L'idée.** Cambridge suit le calendrier nord-américain. Certains mois, il est donc sous une promotion différente de celle des autres magasins, **au même moment de l'année**. La saison est la même pour tous : si la promotion a un effet, Cambridge doit s'écarter des autres ces mois-là.

**Comment tenir compte du contexte propre de Cambridge.** On ne regarde pas son niveau, mais son **écart habituel** aux autres :

1. chaque mois, on calcule l'écart entre Cambridge et la moyenne des autres magasins (en logarithme, ce qui revient à un écart en pourcentage) ;
2. les mois où tout le monde est sous le **même taux** donnent l'écart **habituel**, calculé **exercice par exercice**, parce que Cambridge a moins reculé que les autres (−39,8 % contre −50 % environ, `06`) et que son écart habituel monte d'un exercice à l'autre ;
3. les mois où les taux **diffèrent**, on mesure de combien l'écart s'éloigne de l'écart habituel de l'exercice. C'est une **différence de différences**.

**Le contrôle.** On refait exactement le même calcul pour chacun des autres magasins, comme s'il avait suivi le calendrier de Cambridge. Ces calculs **placebo** donnent l'ampleur des écarts dus au seul hasard (test de permutation).

- **H0** : l'écart de Cambridge aux autres ne dépend pas de la différence de taux.
- **Mesure** : pente de la droite « supplément d'écart ~ différence de taux ». Une pente positive voudrait dire : plus le magasin a de promotion par rapport aux autres, plus il vend par rapport aux autres.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Ventes mensuelles des magasins ouverts pendant les 36 mois
# ---------------------------------------------------------------------------
magasins = ventes[ventes.type == "Magasins"]
# Magasins ouverts au 1er janvier de l'année de référence : présents les 36 mois, mois complets
ouverts_ref = ouverture[ouverture < pd.Timestamp(REFERENCE, 1, 2)].index
complets_36 = [b for b in magasins.id_boutique.unique() if b in ouverts_ref]
mensuel_magasin = (magasins[magasins.id_boutique.isin(complets_36)].groupby(["annee_mois", "id_boutique"])
                   .agg(ca_net=("ca_net", "sum"), unites=("quantite_vente", "sum"), lignes=("id_vente", "size")))
journal = np.log(mensuel_magasin).unstack("id_boutique")       # logarithme : les écarts deviennent des pourcentages
autres_magasins = [b for b in complets_36 if b != ID_TEMOIN]
print(f"Magasin témoin : {NOM_TEMOIN} · magasins de comparaison ouverts les {len(calendrier)} mois : {len(autres_magasins)}")
ecart_taux = calendrier["écart"]
annee_du_mois = calendrier["annee"]
meme_taux = ecart_taux == 0


def supplement(indicateur, boutique):
    """Écart (log) d'une boutique à la moyenne des autres magasins, moins son écart habituel de l'exercice (mois au même taux)."""
    reference = [b for b in autres_magasins if b != boutique]
    ecart = journal[indicateur][boutique] - journal[indicateur][reference].mean(axis=1)
    habituel = ecart[meme_taux].groupby(annee_du_mois[meme_taux]).mean()
    return ecart - annee_du_mois.map(habituel), ecart


# ---------------------------------------------------------------------------
# 2. L'écart de Cambridge, selon la différence de taux
# ---------------------------------------------------------------------------
suivi = pd.DataFrame({"taux_temoin": calendrier[NOM_COURT], "taux_autres": calendrier["Autres boutiques"], "ecart_taux": ecart_taux})
for indicateur in ["ca_net", "unites", "lignes"]:
    suivi[f"supplement_{indicateur}"], suivi[f"ecart_{indicateur}"] = supplement(indicateur, ID_TEMOIN)
habituel_annee = suivi[meme_taux].groupby(annee_du_mois[meme_taux]).ecart_ca_net.mean()
print(f"Écart habituel de {NOM_COURT} aux autres magasins (mois au même taux), par exercice (log) :",
      {a: round(float(v), 2) for a, v in habituel_annee.items()})
suivi["mois"] = calendrier.mois
par_mois_b = suivi[~meme_taux].pivot_table(index=["mois", "taux_temoin", "taux_autres"], columns=annee_du_mois[~meme_taux], values="supplement_ca_net")
par_mois_b.index = [f"{NOMS_MOIS[m - 1]} · {NOM_COURT} {t * 100:.0f} % / autres {a * 100:.0f} %" for m, t, a in par_mois_b.index]
print(f"Supplément d'écart de {NOM_COURT} (chiffre d'affaires, log), les mois où son taux diffère :")
display(par_mois_b.style.format("{:+.2f}"))
par_ecart = suivi.groupby("ecart_taux").agg(mois=("supplement_ca_net", "size"), supplement_ca=("supplement_ca_net", "mean"),
                                            supplement_unites=("supplement_unites", "mean"), supplement_lignes=("supplement_lignes", "mean"))
print("Supplément moyen selon la différence de taux (Cambridge moins les autres) :")
display(par_ecart.style.format({c: "{:+.2f}" for c in par_ecart.columns if c != "mois"}).format_index("{:+.0%}"))
tableaux["7_temoin_par_mois"] = suivi
tableaux["7_temoin_par_ecart"] = par_ecart

# ---------------------------------------------------------------------------
# 3. Pente et test de permutation (placebo sur chacun des autres magasins)
# ---------------------------------------------------------------------------
pentes_b = {}
for indicateur, libelle in [("ca_net", "chiffre d'affaires"), ("unites", "unités vendues"), ("lignes", "nombre de lignes")]:
    pente_temoin = stats.linregress(ecart_taux.values, supplement(indicateur, ID_TEMOIN)[0].values).slope
    pentes_placebo = np.array([stats.linregress(ecart_taux.values, supplement(indicateur, b)[0].values).slope for b in autres_magasins])
    pentes_b[indicateur] = (pente_temoin, pentes_placebo)
    # Valeur p de permutation : part des placebos dont la pente est au moins aussi forte. Le +1 compte le témoin lui-même.
    p_perm = (np.sum(np.abs(pentes_placebo) >= abs(pente_temoin)) + 1) / (len(pentes_placebo) + 1)
    rang = int(np.sum(pentes_placebo < pente_temoin))
    noter_test(registre, f"Angle B ({libelle}) : l'écart de {NOM_COURT} aux autres magasins dépend-il de la différence de taux ?",
               "L'écart aux autres magasins ne dépend pas de la différence de taux", "Permutation (placebo)", len(ecart_taux),
               f"pente = {pente_temoin:+.2f} (placebos de {pentes_placebo.min():+.2f} à {pentes_placebo.max():+.2f}, {rang} plus faibles)", p_perm)

# ---------------------------------------------------------------------------
# 4. Graphique : le supplément d'écart de Cambridge, mois par mois
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12.5, 4.8))
positions = np.arange(len(suivi))
couleurs = np.select([suivi.ecart_taux > 0, suivi.ecart_taux < 0], [viz.ALERTE, viz.PRINCIPALE], viz.GRILLE)
ax.bar(positions, suivi.supplement_ca_net, color=couleurs, edgecolor=viz.GRIS, linewidth=0.5)
ax.axhline(0, color=viz.GRIS, linewidth=1)
etiquettes = [f"{m} · {t * 100:.0f} / {a * 100:.0f}" for m, t, a in zip(suivi.index, suivi.taux_temoin, suivi.taux_autres)]
ax.set_xticks(positions, etiquettes, fontsize=7.5, rotation=60, ha="right")
ax.set_ylabel("Écart à l'écart habituel (log)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:+.1f}".replace(".", ",").replace("+0,0", "0")))
for couleur, libelle in [(viz.ALERTE, f"{NOM_COURT} a plus de promotion que les autres"),
                         (viz.PRINCIPALE, f"{NOM_COURT} a moins de promotion que les autres"), (viz.GRILLE, "Même taux")]:
    ax.bar([0], [0], color=couleur, edgecolor=viz.GRIS, linewidth=0.5, label=libelle)
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.36), frameon=False)
viz.titre(ax, f"Plus de promotion que les autres le même mois : {NOM_COURT} ne vend pas plus",
          f"Chiffre d'affaires de {NOM_COURT} rapporté aux autres magasins, moins son écart habituel de l'exercice · sous chaque mois : son taux / celui des autres (%)")
print("Figure enregistrée :", viz.enregistrer(fig, "07_magasin_temoin"))
plt.show()

**Lecture**

**L'écart habituel** : les mois où Cambridge et les autres magasins sont sous le même taux, Cambridge vend à peu près comme la moyenne en 2019 (+0,02 en logarithme), puis nettement plus en 2020 (+0,28) et en 2021 (+0,16), parce qu'il a moins reculé. C'est son contexte propre, qu'on neutralise exercice par exercice.

**Le supplément d'écart, les mois où son taux diffère** (en logarithme, 0,10 se lit environ 10 % de plus) :

| Mois | Cambridge / autres | 2019 | 2020 | 2021 |
|---|---|---|---|---|
| Octobre | aucune / 20 % | +0,43 | +0,13 | +0,32 |
| Janvier | 5 % / 20 % | −0,16 | −0,03 | +0,19 |
| Avril | aucune / 7 % | −0,24 | −0,05 | −0,05 |
| Juillet | 10 % / aucune | −0,37 | −0,06 | −0,11 |
| Février | 5 % / 7 % | −1,13 | −0,66 | −0,52 |
| Mars | 5 % / 7 % | −0,74 | −0,38 | −0,40 |

- **Quand Cambridge a une promotion et pas les autres** (juillet) : il ne vend pas plus, il recule par rapport à son écart habituel les trois années (−0,37, −0,06, −0,11).
- **Quand Cambridge n'a pas de promotion et que les autres sont à 20 %** (octobre) : il vend nettement mieux que d'habitude les trois années (+0,43, +0,13, +0,32). Au 1er octobre, les autres magasins passent à 20 % et reculent (angle A) ; Cambridge, qui sort au même moment de sa promotion de la rentrée, ne recule pas.
- **En février et mars, Cambridge décroche** (−0,38 à −1,13), alors que son taux (5 %) est presque celui des autres (7 %). Au 1er février, les autres magasins sortent de la promotion à 20 % et bondissent (+62,5 %, +75,1 %, +34,8 %, angle A) ; Cambridge, qui reste sous sa promotion de printemps, ne bondit pas. **Ce rebond de février n'est donc pas un effet de la saison**, puisqu'un magasin soumis à la même saison ne l'a pas : il accompagne la fin de la promotion à 20 %.
- **Les tests** : la pente « supplément ~ différence de taux » est négative sur le chiffre d'affaires (−1,21) et sur le nombre de lignes (−0,57), et elle sort de la plage des 47 magasins placebo (de −0,54 à +0,49 et de −0,11 à +0,13). Mais avec 47 placebos, la plus petite valeur p possible est 1 ÷ 48 = 0,021, au-dessus du seuil corrigé (0,0036) : l'angle ne peut pas être significatif après correction, par construction. Sur les unités vendues, la pente est dans le bruit (+0,14, p = 0,75).
- **Ce que cet angle établit** : à saison égale, un magasin qui a davantage de promotion que les autres **ne vend pas plus**, et les ventes suivent le calendrier de promotion de chaque magasin plutôt que la saison. C'est un indice concordant avec l'angle A, qui écarte en partie sa limite (un effet de saison le même jour). **Sa limite** : un seul magasin, 18 mois sur 36 avec un taux différent, et un écart qui bouge aussi les mois au même taux (jusqu'à ±0,3 en juin 2020 ou décembre 2020, figure) : seuls les écarts de février, mars et octobre, répétés les trois années, sortent nettement de ce bruit.

## 8. Angle C : une ligne remisée est-elle plus grosse ?

**L'idée.** Pendant une promotion, toutes les lignes ne reçoivent pas de remise : environ une sur quatre n'a aucune unité remisée. On compare donc, **pour le même produit, le même mois et dans le même canal**, les lignes remisées aux lignes non remisées. Même produit, même saison, même promotion : seule la remise change.

**La limite, à lire avant le résultat.** Ce lien est en partie **mécanique**. La remise s'applique à une partie des unités d'une ligne : plus une ligne compte d'unités, plus elle a de chances d'en avoir au moins une de remisée. Une ligne remisée sera donc plus grosse en moyenne **même si la remise n'attire aucune vente**. Cet angle donne au mieux une **borne haute** de l'effet. On regarde aussi les unités **payées plein tarif** : si la remise attirait des achats en plus, ces achats seraient remisés ; les unités plein tarif, elles, ne devraient pas être moins nombreuses dans une ligne remisée.

- **H0** : pour un même produit, un même mois et un même canal, les lignes remisées et non remisées ont la même taille.
- **Test** : Wilcoxon apparié sur les couples (produit, mois, canal).

In [ ]:
en_promo["unites_plein_tarif"] = en_promo.quantite_vente - en_promo.quantite_remise
cle_c = ["type", "annee", "id_produit", "annee_mois", "a_remise"]
couples = en_promo.groupby(cle_c)[["quantite_vente", "unites_plein_tarif"]].mean().unstack("a_remise").dropna()
couples.columns = [f"{c}_{'remisee' if r else 'non_remisee'}" for c, r in couples.columns]
couples["rapport"] = np.log(couples.quantite_vente_remisee / couples.quantite_vente_non_remisee)
couples["rapport_plein_tarif"] = couples.unites_plein_tarif_remisee / couples.unites_plein_tarif_non_remisee
_, p_c = stats.wilcoxon(couples.rapport)
ecart_c = np.exp(couples.rapport.median()) - 1
noter_test(registre, "Angle C : une ligne remisée est-elle plus grosse qu'une ligne non remisée du même produit, le même mois ?",
           "Même taille de ligne avec et sans unité remisée", "Wilcoxon apparié", len(couples),
           f"écart médian = {pct(ecart_c, signe=True)} · couples en hausse {pct((couples.rapport > 0).mean(), 0)}, "
           f"en baisse {pct((couples.rapport < 0).mean(), 0)}", p_c)


def resume_couples(df):
    return pd.Series({"couples (produit, mois)": len(df),
                      "taille médiane, ligne non remisée": df.quantite_vente_non_remisee.median(),
                      "taille médiane, ligne remisée": df.quantite_vente_remisee.median(),
                      "couples où la ligne remisée est plus grosse": (df.rapport > 0).mean(),
                      "couples où elle est plus petite": (df.rapport < 0).mean(),
                      "écart moyen sur toutes les unités": np.exp(df.rapport.mean()) - 1,
                      "écart médian sur toutes les unités": np.exp(df.rapport.median()) - 1,
                      "écart médian sur les unités plein tarif": df.rapport_plein_tarif.median() - 1})


resume_c = pd.concat({"Ensemble": resume_couples(couples)}, axis=1).T
resume_c_detail = couples.groupby(level=["type", "annee"]).apply(resume_couples).loc[ORDRE_TYPES]
FORMATS_C = {"couples (produit, mois)": "{:,.0f}", "couples où la ligne remisée est plus grosse": "{:.0%}",
             "couples où elle est plus petite": "{:.0%}", "écart moyen sur toutes les unités": "{:+.1%}", "taille médiane, ligne non remisée": "{:.1f}", "taille médiane, ligne remisée": "{:.1f}",
             "écart médian sur toutes les unités": "{:+.1%}", "écart médian sur les unités plein tarif": "{:+.1%}"}
display(resume_c.style.format(FORMATS_C))
print("Par canal et par exercice :")
display(resume_c_detail.style.format(FORMATS_C))
tableaux["8_lignes_remisees"] = pd.concat([resume_c, resume_c_detail.reset_index().assign(cle=lambda d: d.type + " " + d.annee.astype(str))
                                          .set_index("cle").drop(columns=["type", "annee"])])

**Lecture**

**En magasin, une ligne remisée ne contient pas plus d'unités qu'une ligne non remisée du même produit, le même mois.**

| Couples (produit, mois) | Couples | Taille médiane (non remisée / remisée) | Plus grosse / plus petite | Écart médian | Écart moyen | Unités plein tarif |
|---|---|---|---|---|---|---|
| Magasins 2019 | 11 148 | 8 / 8 | 28 % / 16 % | 0,0 % | +0,7 % | −33,3 % |
| Magasins 2020 | 9 147 | 9 / 9 | 12 % / 19 % | 0,0 % | −1,2 % | −27,8 % |
| Magasins 2021 | 6 666 | 10 / 9 | 11 % / 20 % | 0,0 % | −1,3 % | −25,0 % |
| En ligne 2021 | 4 081 | 12 / 16 | 51 % / 7 % | +20,0 % | +28,5 % | −5,6 % |
| Revendeur 2021 | 2 395 | 12 / 13,5 | 31 % / 3 % | 0,0 % | +16,9 % | −12,5 % |
| Ensemble (trois canaux, trois exercices) | 49 847 | 9 / 9 | 35 % / 12 % | 0,0 % | +12,4 % | −25,0 % |

- **En magasin, l'écart médian est nul les trois exercices**, et l'écart moyen est de −1,2 % à +0,7 %. En 2020 et 2021, la ligne remisée est même plus souvent plus petite que plus grosse. Les unités payées plein tarif y sont moins nombreuses d'un quart à un tiers : **la remise remplace des unités qui auraient été payées plein tarif, elle n'en ajoute pas.**
- **En ligne et chez le revendeur, la ligne remisée est plus grosse** (+20 % à +50 % en médiane en ligne selon l'exercice), avec à peine moins d'unités plein tarif. C'est le cas où le biais mécanique joue le plus : les lignes des entreprises clientes sont grosses (`06b`), et une grosse ligne a plus de chances de contenir une unité remisée. C'est une **borne haute** de l'effet, pas une mesure.
- Le test d'ensemble rejette H0 (Wilcoxon, p < 1e-300), porté par la boutique en ligne et le revendeur.
- Cet angle ne dit rien du **nombre de lignes**, qui fait le chiffre d'affaires d'un magasin (`06`, 76,2 % de la variation) ; or ce nombre ne bouge pas quand la promotion change (angle A).

C'est l'argument que `06` cite (d'après `09_q5_remises`) pour le levier « supprimer les remises en magasin » ; il est confirmé ici sur les trois exercices.

## 9. La corrélation de `06` : la promotion fait-elle baisser le résultat ?

`06` mesure, sur les magasins ouverts toute l'année et pour chaque magasin et chaque mois, une corrélation de Spearman de **−0,44** entre la part des lignes en promotion et le résultat d'exploitation, et de −0,21 avec le chiffre d'affaires. Trois lectures sont possibles :

| Lecture | Ce qu'elle affirme | Trace attendue dans les données |
|---|---|---|
| 1. La promotion fait baisser le résultat | La remise coûte plus qu'elle ne rapporte | Le lien subsiste **à mois égal** : un magasin plus promu que les autres le même mois gagne moins |
| 2. On promeut **là où** ça vend mal | Les magasins faibles reçoivent plus de promotions | La part des lignes en promotion varie d'un magasin à l'autre un mois donné, et elle est plus forte dans les magasins faibles |
| 3. On promeut **quand** ça vend mal, ou quand d'autres charges tombent | Le calendrier des promotions coïncide avec les mois faibles ou chargés | Le lien disparaît à mois égal ; il est porté par quelques mois de l'année |

On teste les trois en décomposant la corrélation.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Reproduire la mesure de 06 : magasins ouverts toute l'année, magasin × mois
# ---------------------------------------------------------------------------
cle_mois = ["id_boutique", "annee", "mois"]
mm = commun[(commun.type == "Magasins") & commun.annee_complete].copy()
part_promo = (ventes[ventes.type == "Magasins"].groupby(cle_mois).agg(part_lignes_en_promotion=("a_promotion", "mean"))).reset_index()
mm = mm.merge(part_promo, on=cle_mois, how="left", validate="one_to_one")
mm["taux_annonce"] = mm.annee_mois.map(calendrier["Autres boutiques"])
mm.loc[mm.id_boutique == ID_TEMOIN, "taux_annonce"] = mm.loc[mm.id_boutique == ID_TEMOIN, "annee_mois"].map(calendrier[NOM_COURT])
r_06 = mm.part_lignes_en_promotion.corr(mm.resultat_exploitation, method="spearman")
r_06_ca = mm.part_lignes_en_promotion.corr(mm.ca_net, method="spearman")
print(f"Table magasin × mois : {nombre(len(mm))} lignes · Spearman avec le résultat d'exploitation : {r_06:+.2f} · avec le CA : {r_06_ca:+.2f}".replace(".", ","))

# ---------------------------------------------------------------------------
# 2. D'où vient la variation de la part des lignes en promotion : des mois ou des magasins ?
# ---------------------------------------------------------------------------
part_mois = eta2(mm.part_lignes_en_promotion, mm.annee_mois)
print(f"Part de la variation de la part des lignes en promotion expliquée par le mois (η²) : {pct(part_mois)}")
print(f"Part expliquée par le magasin : {pct(eta2(mm.part_lignes_en_promotion, mm.id_boutique))}")

# ---------------------------------------------------------------------------
# 3. La corrélation, décomposée
# ---------------------------------------------------------------------------
mm["resultat_avant_publicite"] = mm.marge_nette - mm.frais_generaux
mm["resultat_sans_remise"] = mm.resultat_exploitation + mm.montant_remise
for col in ["part_lignes_en_promotion", "resultat_exploitation", "resultat_avant_publicite", "ca_net"]:
    mm[col + "_a_mois_egal"] = mm[col] - mm.groupby(["annee", "mois"])[col].transform("mean")
decomposition = pd.DataFrame({
    "corrélation (Spearman)": {
        "1. mesure de 06 : part en promotion et résultat d'exploitation": r_06,
        "2. même mesure, en rendant la remise au résultat": mm.part_lignes_en_promotion.corr(mm.resultat_sans_remise, method="spearman"),
        "3. part en promotion et résultat avant publicité": mm.part_lignes_en_promotion.corr(mm.resultat_avant_publicite, method="spearman"),
        "4. part en promotion et publicité du mois": mm.part_lignes_en_promotion.corr(mm.publicite, method="spearman"),
        "5. à mois égal : part en promotion et résultat d'exploitation": mm.part_lignes_en_promotion_a_mois_egal.corr(mm.resultat_exploitation_a_mois_egal, method="spearman"),
        "6. à mois égal : part en promotion et chiffre d'affaires": mm.part_lignes_en_promotion_a_mois_egal.corr(mm.ca_net_a_mois_egal, method="spearman"),
    }})
display(decomposition.style.format("{:+.2f}"))
tableaux["9_decomposition_correlation"] = decomposition
r_mois_egal, p_mois_egal = stats.spearmanr(mm.part_lignes_en_promotion_a_mois_egal, mm.resultat_exploitation_a_mois_egal)
noter_test(registre, "Corrélation de 06 : à mois égal, un magasin plus promu gagne-t-il moins ?",
           "Aucun lien, à mois égal, entre part des lignes en promotion et résultat", "Spearman", len(mm), f"ρ = {r_mois_egal:+.2f}", p_mois_egal)
print("Par exercice :")
for a in ANNEES:
    s = mm[mm.annee == a]
    print(f"  {a} : brute {s.part_lignes_en_promotion.corr(s.resultat_exploitation, method='spearman'):+.2f} · "
          f"à mois égal {s.part_lignes_en_promotion_a_mois_egal.corr(s.resultat_exploitation_a_mois_egal, method='spearman'):+.2f}".replace(".", ","))

# ---------------------------------------------------------------------------
# 4. Les mois : promotion, ventes, publicité et résultat d'un magasin moyen
# ---------------------------------------------------------------------------
profil = mm.groupby("mois").agg(taux_annonce=("taux_annonce", lambda s: s.mode().iloc[0]), part_lignes_en_promotion=("part_lignes_en_promotion", "mean"),
                                ca_net=("ca_net", "mean"), remises=("montant_remise", "mean"), publicite=("publicite", "mean"),
                                frais_generaux=("frais_generaux", "mean"), resultat_exploitation=("resultat_exploitation", "mean"))
profil["indice des ventes"] = profil.ca_net / profil.ca_net.mean() * 100
profil.index = NOMS_MOIS
print("Un magasin ouvert toute l'année, mois moyen des trois exercices :")
display(profil.style.format({"taux_annonce": "{:.0%}", "part_lignes_en_promotion": "{:.0%}", "indice des ventes": "{:.0f}"}
                            | {c: "{:,.0f}" for c in ["ca_net", "remises", "publicite", "frais_generaux", "resultat_exploitation"]}))
tableaux["9_profil_mensuel_magasin"] = profil
groupe_mois = mm.assign(groupe=np.select([mm.mois.isin([1, 2, 3]), mm.taux_annonce > 0], ["promotion, janvier à mars", "promotion, autres mois"], "sans promotion"))
par_groupe = groupe_mois.groupby("groupe").agg(magasins_mois=("ca_net", "size"), ca_net=("ca_net", "mean"), remises=("montant_remise", "mean"),
                                               publicite=("publicite", "mean"), resultat_exploitation=("resultat_exploitation", "mean"))
par_groupe["résultat ÷ CA"] = par_groupe.resultat_exploitation / par_groupe.ca_net
display(par_groupe.style.format({"magasins_mois": "{:,.0f}", "ca_net": "{:,.0f}", "remises": "{:,.0f}", "publicite": "{:,.0f}",
                                 "resultat_exploitation": "{:,.0f}", "résultat ÷ CA": "{:.1%}"}))
r_hors_t1 = groupe_mois[~groupe_mois.mois.isin([1, 2, 3])]
print(f"Sans janvier, février et mars : corrélation brute part en promotion et résultat = "
      f"{r_hors_t1.part_lignes_en_promotion.corr(r_hors_t1.resultat_exploitation, method='spearman'):+.2f}".replace(".", ","))
tableaux["9_resultat_par_groupe_de_mois"] = par_groupe

# ---------------------------------------------------------------------------
# 5. Les trois canaux : résultat ÷ CA selon la promotion du mois (le même calendrier, des publicités très différentes)
# ---------------------------------------------------------------------------
hors_temoin = commun[commun.id_boutique != ID_TEMOIN].copy()
hors_temoin["taux_annonce"] = hors_temoin.annee_mois.map(calendrier["Autres boutiques"])
canaux_taux = hors_temoin.groupby(["type", "taux_annonce"])[["ca_net", "marge_nette", "montant_remise", "publicite", "resultat_exploitation"]].sum()
canaux_taux["taux de marge"] = canaux_taux.marge_nette / canaux_taux.ca_net
canaux_taux["publicité ÷ CA"] = canaux_taux.publicite / canaux_taux.ca_net
canaux_taux["résultat ÷ CA"] = canaux_taux.resultat_exploitation / canaux_taux.ca_net
vue_canaux = canaux_taux[["taux de marge", "publicité ÷ CA", "résultat ÷ CA"]].unstack("type").reorder_levels([1, 0], axis=1)[ORDRE_TYPES]
vue_canaux.index = [libelle_taux(t) for t in vue_canaux.index]
print(f"Selon la promotion du mois, par canal (trois exercices, toutes boutiques sauf {NOM_COURT}) :")
display(vue_canaux.style.format("{:.1%}"))
tableaux["9_resultat_par_canal_et_taux"] = vue_canaux

# ---------------------------------------------------------------------------
# 6. Graphique : la corrélation de 06 vient du calendrier
# ---------------------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [3, 2], "wspace": 0.35})
x = np.arange(12)
ax1.bar(x, profil.resultat_exploitation, color=[COULEURS_TAUX[t] for t in profil.taux_annonce], edgecolor=viz.GRIS, linewidth=0.5)
ax1.plot(x, profil.publicite, color=viz.ALERTE, linewidth=2.2, marker="o", markersize=4, label="Publicité du mois")
ax1.axhline(0, color=viz.GRIS, linewidth=1)
ax1.set_xticks(x, NOMS_MOIS)
viz.format_dollars(ax1, axe="y")
for t in sorted(profil.taux_annonce.unique()):
    ax1.bar([0], [0], color=COULEURS_TAUX[t], edgecolor=viz.GRIS, linewidth=0.5, label=libelle_taux(t).capitalize())
ax1.legend(frameon=False, fontsize=8, loc="lower right", ncol=2)
viz.titre(ax1, "Les mois déficitaires sont ceux de la publicité",
          "Magasin moyen · barres : résultat du mois, selon la promotion · ligne : publicité")
etapes = decomposition["corrélation (Spearman)"].iloc[[0, 2, 4]]
noms_etapes = ["Mesure de 06", "Avant publicité", "À mois égal"]
ax2.barh(noms_etapes[::-1], etapes.values[::-1], color=[viz.GRIS, viz.CATEGORIES[2], viz.ALERTE][::-1])
for y, v in enumerate(etapes.values[::-1]):
    ax2.text(v - 0.01 if v < 0 else v + 0.01, y, f"{v:+.2f}".replace(".", ","), va="center", ha="right" if v < 0 else "left", fontsize=10, color=viz.FONCE)
ax2.axvline(0, color=viz.GRIS, linewidth=1)
ax2.set_xlim(-0.6, 0.1)
ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1f}".replace(".", ",").replace("-0,0", "0,0")))
viz.titre(ax2, "À mois égal, plus aucun lien", "Spearman, part des lignes en promotion et résultat")
print("Figure enregistrée :", viz.enregistrer(fig, "07_correlation_promotion_resultat"))
plt.show()

**Lecture**

**La corrélation de `06` est reproduite** (−0,44 avec le résultat, −0,21 avec le chiffre d'affaires, 1 824 magasins × mois), puis décomposée.

| Mesure | Spearman |
|---|---|
| 1. Part des lignes en promotion et résultat d'exploitation (mesure de `06`) | −0,44 |
| 2. Même mesure, en rendant la remise au résultat | −0,39 |
| 3. Part en promotion et résultat avant publicité | −0,21 |
| 4. Part en promotion et publicité du mois | +0,25 |
| 5. À mois égal : part en promotion et résultat d'exploitation | −0,02 (p = 0,43) |
| 6. À mois égal : part en promotion et chiffre d'affaires | −0,02 |

- **La part des lignes en promotion ne varie que d'un mois à l'autre** : le mois en explique 97,4 % de la variation, le magasin 0,1 %. Un mois donné, tous les magasins ont la même part (sauf Cambridge).
- **Lecture 2, « on promeut là où ça vend mal » : non soutenue.** Aucun magasin ne reçoit plus de promotions qu'un autre.
- **Lecture 1, « la promotion fait baisser le résultat d'un magasin » : non soutenue au niveau du magasin.** À mois égal, le lien disparaît (−0,02, et −0,04 à +0,01 selon l'exercice). La remise elle-même n'en explique qu'une petite partie : la rendre au résultat ne ramène la corrélation que de −0,44 à −0,39.
- **Lecture 3, « on promeut quand le résultat est bas » : retenue.** La corrélation est faite par le calendrier, de deux façons.

| Magasin moyen, par mois (trois exercices) | Chiffre d'affaires | Remises | Publicité | Résultat | Résultat ÷ CA |
|---|---|---|---|---|---|
| Promotion, janvier à mars | 566 k$ | 14 k$ | 388 k$ | −216 k$ | −38,2 % |
| Promotion, autres mois (avril, août à décembre) | 566 k$ | 19 k$ | 27 k$ | +143 k$ | +25,3 % |
| Sans promotion (mai à juillet) | 637 k$ | 0 | 8 k$ | +204 k$ | +32,1 % |

1. **La publicité** : janvier à mars, les trois mois déficitaires des magasins, sont des mois de promotion et concentrent la publicité (346 à 409 k$ par magasin et par mois, `06`). Sans la publicité, la corrélation tombe de −0,44 à −0,21.
2. **La saison** : les mois sans promotion (mai à juillet) sont parmi les meilleurs mois de ventes (indice 106 à 111), la promotion à 20 % tombe sur les plus faibles (octobre à janvier, indice 73 à 92). Même sans janvier à mars, la corrélation brute reste de −0,31 pour cette raison.

**Le même calendrier, trois canaux.**

| Résultat ÷ CA (trois exercices) | Sans promotion | 7 % | 10 % | 20 % |
|---|---|---|---|---|
| En ligne | 39,9 % | 37,1 % | 39,1 % | 36,9 % |
| Revendeur | 39,0 % | 35,2 % | 37,9 % | 35,2 % |
| Magasins | 32,1 % | −13,8 % | 19,1 % | 11,8 % |
| Publicité ÷ CA des magasins | 1,2 % | 44,2 % | 13,0 % | 17,5 % |

En ligne et chez le revendeur, où la publicité pèse moins de 2 % des ventes, un mois de promotion n'est que de 1 à 4 points moins rentable qu'un mois sans promotion, l'ordre de grandeur du taux de marge perdu (57,1 % sans promotion, 55,1 % à 20 %). En magasin, l'écart va jusqu'à 46 points : c'est la publicité, pas la promotion.

**Réponse à la question de `06`** : la corrélation −0,44 ne mesure pas un effet de la promotion sur le résultat d'un magasin. Elle vient de ce qu'on promeut **quand** le résultat est bas : pendant les mois de publicité et sur les mois faibles de l'année. Que les mois de promotion soient faibles **à cause** de la promotion reste une question ouverte à ce niveau ; les angles A et B en donnent des indices.

## 10. Verdict : le registre des tests

Tous les tests du notebook, avec le seuil corrigé (5 % divisé par le nombre de tests).

In [ ]:
verdict_tests = tableau_tests(registre, alpha=ALPHA)
print(f"{len(verdict_tests)} tests · seuil corrigé de Bonferroni : {nombre(ALPHA / len(verdict_tests) , 4)}")
display(verdict_tests.style.format({"n": "{:,.0f}", "p": lambda p: "< 1e-300" if p == 0 else f"{p:.2g}", "seuil corrigé": "{:.4f}"}).hide(axis="index"))
tableaux["10_registre_des_tests"] = verdict_tests

**Lecture du registre**

14 tests, seuil corrigé de Bonferroni : 0,0036.

| Angle | Ce qu'il mesure | Résultat | Solidité | Verdict |
|---|---|---|---|---|
| Ciblage (âge, gamme) | Part des lignes en promotion selon le produit | 71 % à 77 % partout | H0 rejetée, mais V de Cramér 0,016 et 0,007 : effet négligeable | Promotions non ciblées : retenu |
| Comparaison naïve | Jours de promotion contre jours sans promotion | Ventes plus **faibles** sous promotion (−12,9 % par jour en médiane, −23,6 % à 20 %) | Très significatif, mais confondu avec la saison | Ne conclut pas |
| A. Changements de promotion | 14 jours avant et après chaque 1er du mois | **Baisse** quand le taux monte (−9,9 %), hausse quand il baisse (+23,5 %), rien au placebo (+0,2 %) | Lien d'ensemble ρ = −0,50, sous le seuil corrigé ; même sens chaque exercice | Les ventes reculent quand la promotion se renforce : retenu |
| B. Cambridge | Un magasin sous une autre promotion, le même mois | Ne vend pas plus avec plus de promotion ; pas de rebond de février | Pente hors de tous les placebos, mais p = 0,021, au-dessus du seuil par construction | Indice |
| C. Lignes remisées | Même produit, même mois | Aucun écart en magasin ; +20 % à +50 % en ligne | Biais mécanique : borne haute | En magasin, pas de vente en plus : retenu |
| Corrélation de `06` | Part en promotion et résultat, à mois égal | −0,02 | H0 conservée | Effet de la promotion sur le résultat d'un magasin : non soutenu |

**Conclusion sur l'effet des promotions**

1. **Aucun angle ne montre que les promotions augmentent les ventes.** Les deux angles qui mesurent les ventes totales au moment où la promotion change (A et B) vont dans le sens contraire : les ventes baissent quand la promotion se renforce, et la saison ne suffit pas à l'expliquer (Cambridge, soumis à la même saison, ne suit pas le mouvement des autres).
2. **Le seul signal positif est au niveau de la ligne, hors des magasins** (angle C) : en ligne et chez le revendeur, une ligne remisée est plus grosse, mais avec un biais mécanique connu. En magasin, il n'y a pas d'écart.
3. **On ne peut pas chiffrer un effet des promotions avec certitude** : les promotions sont générales et calées sur le calendrier, il n'existe pas de vrai groupe témoin. Mais tous les indices vont dans le même sens : **pas de ventes en plus**.

La conclusion honnête : **l'efficacité des promotions n'est pas démontrée, et les indices disponibles vont contre.** Ce qui suit mesure ce qui, lui, est certain : leur coût.

## 11. Le coût des promotions, et ce qu'elles devraient rapporter

Faute de pouvoir mesurer un gain de ventes, on mesure ce qui est certain : **ce que les remises coûtent**, puis **ce qu'il faudrait qu'elles rapportent** pour être rentables.

**Ce que coûte une remise au résultat.** Rendre une remise *R* augmente le chiffre d'affaires net du même montant, sans changer le coût des produits ; seules les « autres charges », qui suivent les ventes (environ 1,1 % du chiffre d'affaires), augmentent avec lui. Le coût d'une remise pour le résultat d'exploitation est donc *R* × (1 − autres charges ÷ CA). C'est le calcul de `06` pour le levier « supprimer les remises en magasin » (5,42 M$ en 2021), repris ici pour tous les canaux et toutes les promotions.

**Le seuil de rentabilité d'une promotion.** Elle est rentable si les ventes qu'elle ajoute rapportent au moins ce coût. Chaque dollar de vente en plus rapporte *c* de résultat d'exploitation (0,551 $ en magasin en 2021, `06`). Si *V* est le chiffre d'affaires net des lignes en promotion :

> hausse des ventes nécessaire = R × (1 − autres charges ÷ CA) ÷ (c × V)

Exemple : une promotion qui coûte 2 % du chiffre d'affaires qu'elle touche doit le faire progresser d'environ 2 × 0,989 ÷ 55,1 = 3,6 % pour couvrir sa remise. C'est aussi, à l'inverse, la **baisse des ventes qui annulerait le gain** d'une suppression de la remise.

In [ ]:
# ---------------------------------------------------------------------------
# 0. Ce que rapporte un dollar de vente, par canal et par exercice (même calcul que 06, section 8)
# ---------------------------------------------------------------------------
COMPTES_VARIABLES = ["Autres charges"]
variables = reel[reel.nom_compte.isin(COMPTES_VARIABLES)].groupby(["type", "annee"]).montant.sum()
contribution = commun.groupby(["type", "annee"])[["ca_net", "marge_nette"]].sum().join(variables.rename("autres_charges"))
contribution["ce que rapporte 1 $ de vente"] = (contribution.marge_nette - contribution.autres_charges) / contribution.ca_net
c_canal = contribution["ce que rapporte 1 $ de vente"].unstack("annee").loc[ORDRE_TYPES]
ens = commun.groupby("annee")[["ca_net", "marge_nette"]].sum().join(reel[reel.nom_compte.isin(COMPTES_VARIABLES)].groupby("annee").montant.sum())
c_canal.loc["Ensemble"] = (ens.marge_nette - ens.montant) / ens.ca_net
# Part des « autres charges » dans le chiffre d'affaires : rendre une remise augmente le CA net du même montant,
# et donc les autres charges de cette part (même calcul que le levier de 06 : remises × (1 − autres charges ÷ CA))
part_autres_charges = (contribution.autres_charges / contribution.ca_net).unstack("annee").loc[ORDRE_TYPES]
part_autres_charges.loc["Ensemble"] = ens.montant / ens.ca_net
print("Ce que rapporte 1 $ de vente, par canal et par exercice :")
display(c_canal.style.format("{:.3f}"))
print("Autres charges ÷ chiffre d'affaires net, par canal et par exercice :")
display(part_autres_charges.style.format("{:.2%}"))
TAUX_CONTRIBUTION = float(c_canal.loc["Ensemble", EXERCICE])
TAUX_CONTRIBUTION_MAGASINS = float(c_canal.loc["Magasins", EXERCICE])
print(f"Exercice {EXERCICE} : {nombre(TAUX_CONTRIBUTION, 3)} $ par dollar de vente, magasins {nombre(TAUX_CONTRIBUTION_MAGASINS, 3)} $")
tableaux["11_ce_que_rapporte_1_dollar"] = c_canal

# ---------------------------------------------------------------------------
# 1. Le coût des remises, par exercice et par canal
# ---------------------------------------------------------------------------
cout = ventes.groupby(["type", "annee"]).agg(remises=("montant_remise", "sum"), ca_net=("ca_net", "sum"), marge_nette=("marge_nette", "sum"),
                                             ca_net_en_promotion=("ca_net", lambda s: s[ventes.loc[s.index, "a_promotion"]].sum()))
cout = pd.concat([cout.loc[ORDRE_TYPES], pd.concat({"Ensemble": cout.groupby("annee").sum()}, names=["type"])])
cout["remises ÷ CA net"] = cout.remises / cout.ca_net
cout["remises ÷ marge sans remise"] = cout.remises / (cout.marge_nette + cout.remises)
cout["c"] = [c_canal.loc[t, a] for t, a in cout.index]
cout["remises nettes des autres charges"] = [r * (1 - part_autres_charges.loc[t, a]) for (t, a), r in cout.remises.items()]
cout["hausse des ventes nécessaire"] = cout["remises nettes des autres charges"] / (cout.c * cout.ca_net_en_promotion)
print("Coût des remises par canal et par exercice, et hausse des ventes en promotion qui le compenserait :")
display(cout.style.format({"remises": "{:,.0f}", "ca_net": "{:,.0f}", "marge_nette": "{:,.0f}", "ca_net_en_promotion": "{:,.0f}",
                           "remises ÷ CA net": "{:.2%}", "remises ÷ marge sans remise": "{:.2%}", "c": "{:.3f}",
                           "remises nettes des autres charges": "{:,.0f}", "hausse des ventes nécessaire": "{:.1%}"}))
tableaux["11_cout_des_remises"] = cout
remises_annee = cout.loc["Ensemble"].remises
print("Remises, toutes boutiques :", {a: m_dollars(v) for a, v in remises_annee.items()},
      f"· {REFERENCE} → {EXERCICE} : {pct(remises_annee[EXERCICE] / remises_annee[REFERENCE] - 1, signe=True)}")

# ---------------------------------------------------------------------------
# 2. Le seuil, par promotion, sur l'exercice
# ---------------------------------------------------------------------------
ventes_ex = ventes[(ventes.annee == EXERCICE) & ventes.a_promotion]
seuil_promo = ventes_ex.groupby(["type", "taux_remise_annonce"]).agg(remises=("montant_remise", "sum"), ca_net_en_promotion=("ca_net", "sum")).loc[ORDRE_TYPES]
seuil_promo["remises ÷ CA en promotion"] = seuil_promo.remises / seuil_promo.ca_net_en_promotion
canal_promo = seuil_promo.index.get_level_values("type")
seuil_promo["remises nettes des autres charges"] = seuil_promo.remises * (1 - canal_promo.map(part_autres_charges[EXERCICE]).values)
seuil_promo["hausse des ventes nécessaire"] = seuil_promo["remises nettes des autres charges"] / (seuil_promo.ca_net_en_promotion * canal_promo.map(c_canal[EXERCICE]).values)
print(f"Seuil de rentabilité de chaque promotion, exercice {EXERCICE} (toutes les lignes à ce taux, Cambridge compris) :")
display(seuil_promo.style.format({"remises": "{:,.0f}", "ca_net_en_promotion": "{:,.0f}", "remises ÷ CA en promotion": "{:.2%}",
                                  "remises nettes des autres charges": "{:,.0f}", "hausse des ventes nécessaire": "{:.1%}"})
        .format_index(lambda v: v if isinstance(v, str) else f"{v:.0%}"))
tableaux["11_seuil_par_promotion"] = seuil_promo

# Ce qu'on observe aux dates où la promotion commence (angle A), face au seuil des magasins
debut_promo = bascules[(bascules.taux_avant < bascules.taux_apres)]
observe = debut_promo.groupby("taux_apres")["magasins (médiane)"].mean()
comparaison = pd.DataFrame({"hausse nécessaire (magasins, " + str(EXERCICE) + ")": seuil_promo.loc["Magasins", "hausse des ventes nécessaire"],
                            "variation observée quand la promotion commence (magasins, angle A)": observe}).dropna()
comparaison.index = [libelle_taux(t) for t in comparaison.index]
display(comparaison.style.format("{:+.1%}"))
tableaux["11_seuil_contre_observe"] = comparaison

# ---------------------------------------------------------------------------
# 3. Remises et publicité des magasins
# ---------------------------------------------------------------------------
pub_mag = commun[commun.type == "Magasins"].groupby("annee")[["publicite", "montant_remise"]].sum()
pub_mag["publicité ÷ remises"] = pub_mag.publicite / pub_mag.montant_remise
print("Magasins : publicité et remises, par exercice :")
display(pub_mag.style.format({"publicite": "{:,.0f}", "montant_remise": "{:,.0f}", "publicité ÷ remises": "{:.1f}"}))
tableaux["11_publicite_et_remises"] = pub_mag

# ---------------------------------------------------------------------------
# 4. Graphique : la hausse nécessaire et ce qu'on observe
# ---------------------------------------------------------------------------
trace = seuil_promo["hausse des ventes nécessaire"].unstack("type")[ORDRE_TYPES]
fig, ax = plt.subplots(figsize=(10, 4.4))
largeur = 0.22
for i, t in enumerate(ORDRE_TYPES):
    xs = np.arange(len(trace)) + (i - 1) * largeur
    ax.bar(xs, trace[t], width=largeur, color=[viz.CATEGORIES[2], viz.ACCENT, viz.PRINCIPALE][i], label=f"Hausse nécessaire · {t.lower()}")
    for xi, v in zip(xs, trace[t]):
        ax.text(xi, v + 0.002, f"{v * 100:.1f}".replace(".", ","), ha="center", fontsize=8, color=viz.FONCE)
obs = observe.reindex(trace.index)
ax.scatter(np.arange(len(trace)), obs, color=viz.ALERTE, zorder=3, s=60, marker="D", label="Variation observée en magasin au début de la promotion")
for xi, v in zip(np.arange(len(trace)), obs):
    if pd.notna(v):
        ax.text(xi + 0.12, v, f"{v * 100:+.1f}".replace(".", ","), va="center", fontsize=8.5, color=viz.ALERTE)
ax.axhline(0, color=viz.GRIS, linewidth=1)
ax.set_xticks(range(len(trace)), [libelle_taux(t).capitalize() for t in trace.index])
ax.set_yticks(np.arange(-0.12, 0.101, 0.02))
viz.format_pourcent(ax, axe="y")
ax.legend(frameon=False, fontsize=8.5, loc="lower left")
viz.titre(ax, "Il faudrait 2 % à 9 % de ventes en plus pour payer une promotion ; on observe une baisse",
          f"Hausse des ventes en promotion qui couvrirait la remise, exercice {EXERCICE} · losanges : variation des ventes des magasins au début de la promotion")
print("Figure enregistrée :", viz.enregistrer(fig, "07_seuil_rentabilite_promotions"))
plt.show()

**Lecture**

**Un dollar de vente rapporte 0,55 $ dans les trois canaux et les trois exercices** (0,548 à 0,553 $ ; 0,551 $ en magasin en 2021), comme dans `06`.

**Ce que coûtent les remises.**

| Remises | 2019 | 2020 | 2021 | Remises ÷ CA 2021 | Remises ÷ marge sans remise 2021 |
|---|---|---|---|---|---|
| Magasins | 12,4 M$ | 6,4 M$ | 5,5 M$ | 2,08 % | 3,57 % |
| En ligne | 5,6 M$ | 5,5 M$ | 5,3 M$ | 1,96 % | 3,36 % |
| Revendeur | 4,0 M$ | 3,6 M$ | 3,1 M$ | 2,01 % | 3,46 % |
| **Ensemble** | **22,1 M$** | **15,5 M$** | **13,9 M$** | 2,02 % | 3,46 % |

Nettes des autres charges (1,07 % du chiffre d'affaires des magasins en 2021), les remises de 2021 coûtent au résultat **13,74 M$**, dont **5,42 M$ en magasin** (le chiffre du levier de `06`), 5,21 M$ en ligne et 3,11 M$ chez le revendeur.

**Ce qu'elles devraient rapporter.** Pour couvrir leur remise, les promotions de 2021 devraient faire progresser les ventes qu'elles touchent de **5,1 %** (magasins 5,2 %, en ligne 5,0 %, revendeur 5,1 % ; magasins 5,9 % en 2019, 5,2 % en 2020). Par promotion, en magasin, en 2021 (toutes les lignes à ce taux, Cambridge compris) :

| Promotion (magasins, 2021) | Remises | Remises nettes des autres charges | Chiffre d'affaires en promotion | Hausse des ventes nécessaire | Variation observée au début de la promotion (angle A) |
|---|---|---|---|---|---|
| 7 % (février à avril) | 0,58 M$ | 0,57 M$ | 65,0 M$ | +1,6 % | (la promotion commence après celle à 20 %) |
| 10 % (août, septembre) | 0,98 M$ | 0,97 M$ | 45,0 M$ | +3,9 % | −9,5 % |
| 20 % (octobre à janvier) | 3,91 M$ | 3,87 M$ | 76,7 M$ | +9,2 % | −11,1 % |
| 5 % (Cambridge) | 0,01 M$ | 0,01 M$ | 1,1 M$ | +1,9 % | |
| **Total des magasins** | **5,48 M$** | **5,42 M$** | **187,7 M$** | **+5,2 %** | |

Les seuils sont les mêmes dans les trois canaux (1,5 % à 9,2 %). **La promotion à 20 % représente 71 % des remises des magasins** (3,87 M$ sur 5,42 M$, nets des autres charges) et devrait faire vendre 9,2 % de plus ; au moment où elle commence, les ventes des magasins reculent de 11,1 %. **Sur la base des données disponibles, les remises sont un coût sans contrepartie mesurable.**

**Remises et publicité.** En magasin, la publicité coûte 6,6 fois les remises en 2019, 10,9 fois en 2020 et 11,3 fois en 2021 (61,9 M$ contre 5,5 M$). Pour le résultat des magasins, l'enjeu de la publicité est d'un ordre de grandeur supérieur (`06`).

## 12. Les leviers sur l'exercice 2021

On chiffre ce que rapporterait la suppression des remises, par canal et par promotion, à ventes inchangées, puis selon la réaction des ventes. **Le choix du niveau de remise à garder** (quelle part des unités, quel taux, quel produit) relève de `09_q5_remises` : ce notebook fournit le coût et le seuil.

In [ ]:
commun_ex = commun[commun.annee == EXERCICE]
resultat_an = commun_ex.resultat_exploitation.sum()
resultat_magasins_an = commun_ex[commun_ex.type == "Magasins"].resultat_exploitation.sum()
leviers = []


def ajouter_levier(nom, canal, remises, gain, ventes_exposees, commentaire):
    """Un levier : remises supprimées, gain à ventes inchangées (remises nettes des autres charges, comme dans 06),
    et baisse des ventes exposées qui annule le gain (gain ÷ (ventes × c du canal))."""
    c = c_canal.loc[canal if canal in c_canal.index else "Ensemble", EXERCICE]
    leviers.append({"levier": nom, "canal": canal, "remises supprimées": remises, "gain par an": gain, "hausse du résultat": gain / resultat_an,
                    "résultat des magasins après": resultat_magasins_an + gain if canal == "Magasins" else np.nan,
                    "ventes exposées": ventes_exposees, "baisse des ventes qui annule le gain": gain / (ventes_exposees * c),
                    "commentaire": commentaire})


for t in ORDRE_TYPES:
    s = cout.loc[(t, EXERCICE)]
    ajouter_levier(f"Supprimer les remises · {t.lower()}", t, s.remises, s["remises nettes des autres charges"], s.ca_net_en_promotion,
                   "Aucun angle ne montre de hausse des ventes sous promotion")
s = cout.loc[("Ensemble", EXERCICE)]
ajouter_levier("Supprimer les remises · toutes boutiques", "Ensemble", s.remises, s["remises nettes des autres charges"], s.ca_net_en_promotion,
               "Somme des trois canaux")
for taux in [0.20, 0.10, 0.07]:
    s = seuil_promo.loc[("Magasins", taux)]
    ajouter_levier(f"dont : supprimer la {libelle_taux(taux)} en magasin", "Magasins", s.remises, s["remises nettes des autres charges"],
                   s.ca_net_en_promotion, "Partie de « supprimer les remises · magasins », ne s'y ajoute pas")
leviers = pd.DataFrame(leviers).set_index("levier")
print(f"Résultat d'exploitation {EXERCICE} : {m_dollars(resultat_an)} · magasins : {m_dollars(resultat_magasins_an)}")
display(leviers.style.format({"remises supprimées": "{:,.0f}", "gain par an": "{:,.0f}", "hausse du résultat": "{:+.1%}",
                              "résultat des magasins après": "{:,.0f}", "ventes exposées": "{:,.0f}",
                              "baisse des ventes qui annule le gain": "{:.1%}"}, na_rep="sans objet"))
tableaux["12_leviers"] = leviers

# ---------------------------------------------------------------------------
# Grille : résultat des magasins si l'on supprime leurs remises, selon la réaction des ventes en promotion
# ---------------------------------------------------------------------------
remises_mag = cout.loc[("Magasins", EXERCICE), "remises nettes des autres charges"]
ventes_promo_mag = cout.loc[("Magasins", EXERCICE), "ca_net_en_promotion"]
reactions = [0.02, 0, -0.01, -0.02, -0.03, -0.05]
grille = pd.DataFrame({"résultat des magasins": [resultat_magasins_an + remises_mag + r * ventes_promo_mag * TAUX_CONTRIBUTION_MAGASINS for r in reactions]},
                      index=[f"ventes en promotion {pct(r, 0, signe=True)}" if r else "ventes inchangées" for r in reactions])
grille["écart au résultat actuel"] = grille["résultat des magasins"] - resultat_magasins_an
print(f"Magasins, {EXERCICE} : résultat ({m_dollars(resultat_magasins_an)} aujourd'hui) si l'on supprime les remises, selon la réaction des ventes :")
display(grille.style.format("{:,.0f}").map(lambda v: f"color: {viz.ALERTE}" if v < 0 else ""))
tableaux["12_grille_suppression"] = grille

**Lecture**

**Les leviers, chiffrés sur l'exercice 2021, à ventes inchangées.** Le gain est la remise supprimée nette des autres charges, comme dans `06`. Les trois lignes « dont » détaillent la suppression des remises en magasin, promotion par promotion : elles en font **partie** et ne s'y ajoutent pas (3,87 + 0,97 + 0,57 + 0,01 pour Cambridge = 5,42 M$).

| Levier | Remises supprimées | Gain par an | Hausse du résultat | Résultat des magasins après | Baisse des ventes en promotion qui annule le gain |
|---|---|---|---|---|---|
| Supprimer les remises, toutes boutiques | 13,90 M$ | 13,74 M$ | +9,7 % | | 5,1 % |
| Supprimer les remises en ligne | 5,27 M$ | 5,21 M$ | +3,7 % | | 5,0 % |
| Supprimer les remises chez le revendeur | 3,14 M$ | 3,11 M$ | +2,2 % | | 5,1 % |
| **Supprimer les remises en magasin** (levier de `06`) | **5,48 M$** | **5,42 M$** | **+3,8 %** | **−5,4 M$** | **5,2 %** |
| dont : promotion à 20 % en magasin | 3,91 M$ | 3,87 M$ | +2,7 % | −6,9 M$ | 9,2 % |
| dont : promotion à 10 % en magasin | 0,98 M$ | 0,97 M$ | +0,7 % | −9,8 M$ | 3,9 % |
| dont : promotion à 7 % en magasin | 0,58 M$ | 0,57 M$ | +0,4 % | −10,2 M$ | 1,6 % |

**Si les ventes réagissent** : supprimer les remises des magasins les laisse à −5,4 M$ si les ventes en promotion ne bougent pas, à −3,3 M$ si elles progressent de 2 % (ce que suggère l'angle A), à −7,4 M$ si elles reculent de 2 %. Le gain disparaît au-delà de 5,2 % de ventes en promotion perdues (−10,5 M$ à −5 %).

**Comment lire ce tableau.**

- **Dans la suppression des remises en magasin, la part la plus sûre est la promotion à 20 %** : 3,87 M$ des 5,42 M$, et il faudrait que sa suppression fasse perdre plus de 9,2 % des ventes d'octobre à janvier pour qu'elle soit une erreur ; au contraire, les ventes reculent quand elle commence. C'est la première étape possible, pas un gain en plus.
- **La promotion à 7 % coûte peu** (0,57 M$) mais son seuil est bas (1,6 %) : c'est la seule dont les jours ne vendent pas moins que les jours sans promotion (section 5). La garder ou la supprimer change peu le résultat.
- **En ligne et chez le revendeur**, la remise va avec des lignes plus grosses (angle C) ; une partie des clients sont des entreprises (`06b`). Supprimer la remise y est plus risqué qu'en magasin : à tester avant de généraliser.
- **Aucun de ces leviers ne rend les magasins bénéficiaires à lui seul** (−5,4 M$ au mieux à ventes inchangées). Il s'ajoute aux leviers de charges de `06` (frais généraux, publicité), bien plus importants (synthèse, section 13).
- Le niveau de remise à garder (quelle part des unités, quel taux, quel produit) est chiffré dans `09_q5_remises` ; ce notebook fournit le coût et le seuil.

## 13. Synthèse : réponse à la question Q2

**Question Q2** : *les promotions sont-elles efficaces ? Augmentent-elles les ventes, et à quel coût ?*

### 1. Réponse

**Non démontré, et les indices vont contre.** Les promotions s'appliquent à tout (tous les produits, toutes les boutiques, neuf mois sur douze), selon un calendrier reconduit à l'identique de 2019 à 2021. Aucun angle ne montre qu'elles font vendre davantage ; les ventes reculent même quand la promotion se renforce. Les remises représentent 13,9 M$ en 2021 (22,1 M$ en 2019), soit 2 % du chiffre d'affaires et 3,5 % de la marge ; nettes des autres charges, elles coûtent 13,7 M$ au résultat, dont 5,42 M$ en magasin. Il faudrait qu'elles fassent vendre 5,1 % de plus pour se payer.

**Mais les promotions ne sont pas un problème prioritaire.** Leur poids sur le résultat est bien plus faible que celui des trois problèmes établis par `06` et `06b` : la hausse des frais généraux, le non-renouvellement de l'assortiment et la perte des grands comptes de la boutique en ligne. Les retours (`08_q3_q4_retours`) pèsent encore moins.

| Problème (exercice 2021) | Ordre de grandeur | Levier chiffré | Source |
|---|---|---|---|
| Assortiment non renouvelé (552, 429 puis 272 lancements) | les nouveautés de l'année portent 69 % de la chute des ventes des magasins | 46,1 M$ (rythme de lancement de 2019) | `06`, `06b` |
| Hausse des frais généraux en deux exercices | +26,3 M$ pour l'entreprise (155,0 → 181,3 M$), dont +12,9 M$ en magasin (83,9 → 96,8 M$) | 9,2 M$ (personnel, administration, informatique au montant de 2019) | `06` |
| Perte des grands comptes de la boutique en ligne | ventes aux entreprises clientes historiques : 169,0 M$ (2019), 86,6 M$ (2020), 0,1 M$ (2021) | | `06b` (table des ventes en ligne détaillée, ne s'additionne pas avec les autres montants) |
| Publicité des magasins | 61,9 M$, 11,3 fois les remises des magasins | 6,2 M$ (−10 %) | `06`, ce notebook |
| **Promotions et remises** | 13,9 M$ de remises, 13,7 M$ nets des autres charges | **5,42 M$** (supprimer les remises en magasin) | `06`, ce notebook |
| Retours | détaillés dans `08_q3_q4_retours` | 2,62 M$ (réduire les retours de moitié) | `06`, `08_q3_q4_retours` |

Pour un résultat d'exploitation de 141,7 M$ (magasins −10,8 M$, 4 magasins bénéficiaires sur 53), le levier des remises en magasin vaut 3,8 % du résultat, contre 46,1 M$ pour l'assortiment et 9,2 M$ pour les frais généraux. Les leviers de ce notebook restent valables, mais **secondaires** : ils ne rendent pas les magasins bénéficiaires et ne traitent pas la cause de la chute des ventes.

| | 2019 | 2020 | 2021 |
|---|---|---|---|
| Part des lignes en promotion | 75,2 % | 74,0 % | 73,9 % |
| Taux annoncé moyen · taux effectif | 12,8 % · 3,04 % | 12,7 % · 2,72 % | 13,1 % · 2,73 % |
| Remises, toutes boutiques | 22,1 M$ | 15,5 M$ | 13,9 M$ |
| Remises des magasins | 12,4 M$ | 6,4 M$ | 5,5 M$ |
| Remises des magasins, nettes des autres charges | 12,3 M$ | 6,4 M$ | 5,4 M$ |
| Hausse des ventes nécessaire pour couvrir la remise (magasins) | 5,9 % | 5,2 % | 5,2 % |

### 2. Registre des constats

| Constat | Verdict |
|---|---|
| Les promotions sont générales et suivent le même calendrier les trois exercices ; seul Cambridge suit un autre calendrier | Retenu |
| Le taux effectif est très inférieur au taux annoncé (4,70 % pour une promotion « à 20 % » en 2021) et baisse de 10 % depuis 2019 parce que le panier grossit | Retenu |
| Les promotions portent autant sur les nouveautés que sur les produits anciens, sur toutes les gammes : elles n'accompagnent ni le renouvellement de l'assortiment, ni la descente en gamme | Retenu |
| Les promotions augmentent les ventes | Non soutenu |
| Les ventes reculent quand le taux de promotion monte, et repartent quand il baisse (angle A) | Retenu |
| À saison égale, un magasin plus promu que les autres ne vend pas plus (Cambridge) | Indice |
| En magasin, une ligne remisée ne contient pas plus d'unités qu'une ligne non remisée | Retenu |
| En ligne et chez le revendeur, la remise fait grossir la ligne | Ouvert (biais mécanique, borne haute de +20 % à +50 %) |
| La corrélation −0,44 de `06` vient de ce que la promotion fait baisser le résultat d'un magasin | Non soutenu (−0,02 à mois égal) |
| On promeut là où ça vend mal (certains magasins plus que d'autres) | Non soutenu |
| On promeut quand le résultat est bas (mois de publicité, mois faibles de l'année) | Retenu |
| Les mois de promotion sont faibles à cause de la promotion | Ouvert (indices des angles A et B, pas de groupe témoin) |
| Les remises sont un coût sans contrepartie mesurable | Retenu sur les données disponibles |
| Les promotions ne sont pas un problème prioritaire : leur levier (5,42 M$ en magasin) est bien plus petit que ceux de l'assortiment (46,1 M$) et des frais généraux (9,2 M$) | Retenu (chiffres de `06` et `06b`) |

### 3. Recommandations chiffrées sur l'exercice 2021

1. **Supprimer les remises en magasin** : +5,42 M$ par an à ventes inchangées (résultat des magasins de −10,8 à −5,4 M$), gain annulé au-delà de 5,2 % de ventes en promotion perdues. Le niveau de remise à conserver éventuellement est chiffré dans `09_q5_remises`.
2. **Commencer par la promotion à 20 % en magasin** (octobre à janvier), en la supprimant ou en la suspendant à titre d'essai : 3,87 M$ des 5,42 M$ du point 1 (une partie de ce levier, pas un gain en plus). Il faudrait perdre plus de 9,2 % des ventes de la période pour que ce soit une erreur ; les ventes reculent de 11,1 % quand elle commence.
3. **Tester avant de généraliser, en ligne et chez le revendeur** (8,4 M$ de remises en 2021) : appliquer la promotion à une partie des produits ou des clients seulement, et garder les autres comme groupe témoin. C'est la seule façon de mesurer l'effet d'une promotion, et la recommandation la plus importante de ce notebook pour l'avenir.
4. **Si une promotion est gardée, la mettre au service de l'assortiment** : réserver la remise aux nouveautés au lancement, ou au déstockage des produits qui ne tournent pas (`06b`), plutôt qu'à tout le catalogue.
5. **Garder ces leviers à leur rang** : ils sont secondaires. Les priorités sont le renouvellement de l'assortiment (46,1 M$), les frais généraux (9,2 M$) et la publicité des magasins (6,2 M$ pour −10 %, 11,3 fois les remises), chiffrés dans `06`, et la perte des grands comptes en ligne (`06b`).

### 4. Points à valider avec les gens du métier

- l'objectif commercial des promotions saisonnières (faire vendre, fidéliser, suivre la concurrence, écouler du stock) : si ce n'est pas la hausse des ventes, le critère de rentabilité utilisé ici n'est pas le bon ;
- pourquoi une ligne remisée ne compte qu'environ trois unités remisées, quelle que soit sa taille (règle de caisse, plafond par client) : `09` ;
- la possibilité d'un test piloté (magasins ou produits témoins) sur une saison ;
- pour Cambridge, la raison du calendrier nord-américain, et s'il peut servir de magasin témoin à l'avenir ;
- l'effet des remises sur les achats des entreprises clientes en ligne, que nos données ne séparent pas des particuliers.

### 5. Limites

- **Pas de groupe témoin** : promotion et saison sont confondues ; les conclusions sur l'effet reposent sur trois méthodes indirectes et sur leur accord.
- **Un seul magasin témoin** : l'angle B ne peut pas être significatif après correction.
- **Pas de client ni de ticket** : une ligne est un cumul journalier ; on ne sait pas si une promotion attire de nouveaux acheteurs ou fait seulement acheter plus tôt.
- **Charges mensuelles** : la rentabilité se compare entre mois, pas entre jours de promotion.

## Enregistrement des tableaux

In [ ]:
# Tous les tableaux de ce notebook dans un classeur Excel, un onglet par tableau,
# pour les reprendre dans le rapport d'analyse (livrable L4) sans relancer le code.
tableaux_excel = {}
for onglet, table in tableaux.items():
    table = table.copy()
    if isinstance(table, pd.Series):
        table = table.to_frame()
    # Excel n'accepte pas les dates avec fuseau ni les index multiples de type période : on écrit l'index en texte si besoin
    if isinstance(table.index, pd.DatetimeIndex):
        table.index = table.index.strftime("%Y-%m-%d")
    tableaux_excel[onglet] = table
chemin_excel = analyse.exporter_tableaux(tableaux_excel, config.L4 / "q2_tableaux.xlsx")
print("Classeur enregistré :", chemin_excel)
print("Onglets :", list(tableaux_excel))